# تحلیل ماشینی سطح محصول: کی‌پاپ ۱۰۵ و ایندی کره‌ای ۳۳ (نسخه ۲.۰، مستندسازی‌شده)

این نوتبوک همه‌ی محاسبه‌های سمت محصول رساله را از ویدئوی خام تا جدول‌های فصل ۴ و محرک‌های مطالعه‌ی ردیابی چشم انجام می‌دهد. سلول‌ها به همان ترتیب منطقی خط تولید داده مرتب شده‌اند. تحلیل داده‌ی نگاه در نوتبوک جداگانه‌ی `KPOP_Gaze_Analysis_v1.ipynb` است.

## منطق علمی در یک نگاه
- **سازه.** استانداردسازی در معنای آدورنو (۱۹۴۱) ویژگی رابطه‌ای محصول است: هر محصول تا چه اندازه به طرح مشترک مجموعه نزدیک است. این نوتبوک آن را «مرکزیت نسبی در فضای بازنمایی یک مدل ویدئو» عملیاتی می‌کند. این برچسب تا پیش از وارسی روایی با کدگذاری انسانی (V1، سلول ۱۸) یک برچسب عملیاتی است.
- **بازنمایی.** مدل از پیش آموزش‌دیده‌ی InternVideo2-CLIP-S (`OpenGVLab/InternVideo2_CLIP_S`، commit ثبت‌شده در سلول ۱۲) با تنظیمات منجمد نسخه‌ی ۱.۰: پنجره‌های ۸ ثانیه‌ای با گام ۴ ثانیه، ۸ فریم، پیش‌پردازش رسمی (۲۲۴ پیکسل، میانگین و انحراف ImageNet). بردار ویدئو میانگین نرمال‌شده‌ی بردار پنجره‌هاست (ترتیب زمانی در آن حفظ نمی‌شود).
- **شاخص اصلی.** شباهت کسینوسی متوازن: نیمی میانگین شباهت به ویدئوهای گروه‌های زن و نیمی به گروه‌های مرد، بدون خود ویدئو، سپس z. توازن جنسیتی لازم است چون ۷۷ ویدئو از ۱۰۵ ویدئو زن‌اند و شاخص خام شکاف جنسیتی ساختگی می‌سازد. مقدار مطلق شباهت تفسیر نمی‌شود (مخروط باریک فضای تعبیه؛ Liang et al., 2022)، فقط رتبه و z.
- **نسخه‌های حساسیت.** شاخص خام، درون‌جنسیتی، k نزدیک‌ترین همسایه و سطح پنجره.
- **تحلیل محتوای کمّی خودکار.** OpenCV (روشنایی، کنتراست، رنگ‌مایه، حرکت)، PySceneDetect (برش و طول نما)، YuNet (چهره و نما‌ی نزدیک).
- **MIR.** librosa: تمپو و ضرب، کروما، MFCC، انرژی، تکرار صوتی، نامزدهای کورس؛ سازه‌های آدورنویی و شبه‌فردیت‌سازی.
- **پیکره‌ی مرجع.** ایندی کره‌ای (۳۳ ویدئو از ۳۰ هنرمند) با همان مسیر پردازش؛ مرجع توصیفی، نه گروه کنترل.
- **تکرارپذیری.** هر مرحله برای هر ویدئو جدا در `analysis_v2_0/per_video/<corpus>/<video>/` ذخیره و در اجرای بعد رد می‌شود؛ آزمون دود (سلول ۷) بازتولید بردارهای اجرای اول را می‌سنجد؛ نسخه‌ی مدل و کتابخانه‌ها در سلول ۱۲ ثبت می‌شود؛ همه‌ی بذرهای تصادفی ثابت‌اند.

## سلول‌ها
| سلول | کار | ورودی | خروجی | کی اجرا شود |
|---|---|---|---|---|
| ۱ | بررسی دستگاه (پایتون، torch، CPU) | — | — | هر بار |
| ۲ | نصب کتابخانه‌ها با نسخه‌های ثابت | — | — | بار اول یا پس از تغییر محیط |
| ۳ | مسیرها | پوشه‌ی `KPOP` | پوشه‌های خروجی | هر بار |
| ۴ | تنظیمات و توابع منجمد نسخه‌ی ۱.۰ و بارگذاری مدل | — | مدل در حافظه | هر بار |
| ۵ | نمایه‌ی نقطه‌های ذخیره‌ی اجرای اول | پوشه‌ی اجرای اول | نگاشت نام فایل | هر بار |
| ۶ | توابع استخراج یک‌باره‌ی هر ویدئو (بردار، تصویر، برش، چهره، صدا، برگه‌ی تماس) | ویدئوها | `per_video/` | هر بار (فقط تعریف) |
| ۷ | آزمون دود: استخراج دوباره‌ی یک ویدئو و مقایسه با اجرای اول | یک ویدئو | گزارش | فقط بار اول |
| ۸ | اجرای اصلی روی ۱۰۵ ویدئو | پیکره‌ی کی‌پاپ | `per_video/kpop/` | یک بار (چند ساعت) |
| ۹ | تحلیل کی‌پاپ: شاخص‌ها، رقیب‌ها، دو تیپ جنسیتی، روند زمانی، ویژگی‌ها، نمونه‌ی کدگذاری | `per_video/kpop` | `tables/kpop_product_analysis.xlsx`، برگه‌ی کدگذاری کور | پس از ۸ |
| ۱۰ | پیکره‌ی ایندی: همان استخراج و مقایسه‌ی بصری با کی‌پاپ | پوشه‌ی ایندی | `tables/indie_reference_comparison.xlsx` | پس از ۹ |
| ۱۱ | MIR، نوع‌وارگی ویژگی‌ها، ابعاد اکتشافی | خروجی ۸ و ۱۰ | `tables/mir_and_typicality.xlsx` | پس از ۱۰ |
| ۱۲ | وارسی‌های داوران: جدایی‌پذیری با کنار گذاشتن کل گروه، کی‌پاپ ۲۰۲۱ به بعد، تاکردن تمپو، ثبت نسخه‌ها | خروجی ۹ تا ۱۱ | `tables/reviewer_checks.xlsx` | پس از ۱۱ |
| ۱۳ | جمع‌بندی سمت محصول با قاعده‌های فصل ۳ | خروجی ۹ تا ۱۲ | `tables/FINAL_product_results.xlsx` | پس از ۱۲ |
| ۱۴ | شاخص قطعه‌ی کورس و انتخاب ۸ قطعه‌ی کی‌پاپ | `chorus_shortlist.xlsx` | `tables/eyetracking_selection.xlsx` | پس از بازبینی دستی کورس‌ها؛ **انتخاب قفل شده است، دوباره اجرا نشود** |
| ۱۵ | کلیپ‌های داوری کور (R01، …) برای صفحه‌ی داوری انسانی | شورت‌لیست | `web_clips/rating/`، `rate_key.xlsx` (محرمانه) | یک بار |
| ۱۶ | کلیپ‌های سایت C1 تا C8 (۱۰۸۰، بلندی یکسان، نام بی‌نشان) | انتخاب ۸ قطعه | `web_clips/clips/`، `clip_key.xlsx` (محرمانه) | یک بار |
| ۱۷ | چهار قطعه‌ی ایندی C9 تا C12 و شاخص هر ۱۲ قطعه در یک مقیاس | `indie_chorus.xlsx` | `tables/eyetracking_selection_12.xlsx` | یک بار |
| ۱۸ | پس از کدگذاری انسانی: پایایی (آلفای کریپندورف) و روایی شاخص (V1) | برگه‌های دو کدگذار | جدول روایی | پس از کدگذاری |

## نکته‌ها
- سلول‌های ۱ تا ۶ پیش‌نیاز همه‌اند. سلول‌های ۷، ۸، ۱۴ تا ۱۷ کارهای یک‌باره‌اند و نتیجه‌شان روی دیسک است؛ برای بازتولید جدول‌ها کافی است ۱ تا ۶ و سپس ۹ تا ۱۳ اجرا شوند.
- فایل‌های کلید (`clip_key.xlsx`، `rate_key.xlsx`) سطح هر قطعه را نشان می‌دهند و نباید منتشر شوند.
- خروجی سلول‌ها در این نسخه پاک شده‌اند تا فایل سبک و بدون اطلاعات محرمانه باشد؛ نتیجه‌ها در پوشه‌ی `tables` هستند.


In [ ]:
# 1) Local machine check (no Colab, no Drive mount)
import os, sys, torch
print('Python', sys.version.split()[0], '| torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
torch.set_num_threads(max(1, (os.cpu_count() or 4) - 2))
print('CPU threads used:', torch.get_num_threads())

In [ ]:
# 2) Install (same transformers version as the frozen v1.0 run)
%pip install -q "transformers==4.51.3" "huggingface_hub<1.0" decord openpyxl xlsxwriter pandas numpy tqdm scenedetect librosa open_clip_torch timm einops matplotlib scipy statsmodels psutil imageio-ffmpeg
print('installed')

In [ ]:
# 3) Paths  >>> only edit if your folders are elsewhere <<<
from pathlib import Path
BASE        = Path.home() / 'KPOP'                                                   # e.g. C:/Users/<you>/KPOP
KPOP_DIR    = BASE / 'kpop Corpus'                                                    # 105 K-pop MVs
INDIE_DIR   = BASE / 'Kindie Corp' if (BASE / 'Kindie Corp').exists() else BASE / 'kindie_30'   # Korean indie MVs
OLD_CKPT    = BASE / 'mv_checkpoints_v1_0'   # not used
# folder of the FIRST real run (its 'embeddings' subfolder is used to check that the new extraction reproduces it)
_c = sorted(BASE.glob('*InternVideo2_Analysis'))
OLD_ANALYSIS = _c[0] if _c else None
METADATA    = BASE / 'MV_metadata_105_verified_v2.xlsx'     # upload the v2 Excel here
OUT         = BASE / 'analysis_v2_0'                         # everything new goes here

VIDEO_EXT = ('.mp4', '.mkv', '.webm', '.mov')
def list_videos(d):
    # the high-resolution copies for the eye-tracker (sub-folder '1080') are not part of the corpus
    return sorted(p for p in Path(d).rglob('*') if p.suffix.lower() in VIDEO_EXT and not p.name.startswith('.') and '1080' not in p.parent.name)
for d in ['per_video', 'tables', 'figures', 'smoke_test']:
    (OUT / d).mkdir(parents=True, exist_ok=True)
print('K-pop folder exists:', KPOP_DIR.exists(), '| videos:', len(list_videos(KPOP_DIR)) if KPOP_DIR.exists() else 0)
print('old v1.0 checkpoints:', len(list(OLD_CKPT.glob('*.npz'))) if OLD_CKPT.exists() else 'folder not found (all videos will be extracted)')
print('metadata found:', METADATA.exists())
print('first-run folder:', OLD_ANALYSIS)

In [ ]:
# 4) Frozen v1.0 settings and functions (copied unchanged from InternVideo2_Kpop_Main100_Frozen_v1_0)
import types, importlib.machinery, re, json, gc, hashlib, math, time, shutil, subprocess, urllib.request
import numpy as np, pandas as pd, cv2
import torch.nn.functional as F
from decord import VideoReader, cpu
from tqdm.auto import tqdm

def install_flash_attn_stub():
    # FlashAttention is disabled in the model config; these empty modules only let the model code import.
    class _Unused:
        def __init__(self, *a, **k): raise RuntimeError('flash_attn stub called: FlashAttention must stay disabled')
    def _getattr(name):
        if name.startswith('__'): raise AttributeError(name)
        return _Unused
    for name in ['flash_attn', 'flash_attn.flash_attn_interface', 'flash_attn.bert_padding', 'flash_attn.modules',
                 'flash_attn.modules.mlp', 'flash_attn.ops', 'flash_attn.ops.rms_norm', 'flash_attn.ops.fused_dense', 'flash_attn.layers',
                 'flash_attn.layers.rotary']:
        m = types.ModuleType(name); m.__path__ = []
        m.__spec__ = importlib.machinery.ModuleSpec(name, loader=None, is_package=True); m.__getattr__ = _getattr
        sys.modules[name] = m
install_flash_attn_stub()

MODEL_ID = 'OpenGVLab/InternVideo2_CLIP_S'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32   # CPU needs float32
NUM_FRAMES, CLIP_SECONDS, STRIDE_SECONDS, EMB_DIM = 8, 8.0, 4.0, 512

def make_clip_starts(duration, clip_seconds=CLIP_SECONDS, stride_seconds=STRIDE_SECONDS):
    # Rule of the FIRST REAL RUN (verified from its checkpoints: 198.24 s -> 50 windows, 186.98 s -> 47 windows):
    # a window starts every 4 s until the end of the video; windows near the end are clipped at the video end.
    return [i * stride_seconds for i in range(max(1, math.ceil(duration / stride_seconds)))]

def sample_video_clip(vr, fps, start_sec, end_sec, num_frames=8):
    n = len(vr)
    start_frame = max(0, int(round(start_sec * fps))); end_frame = min(n, int(round(end_sec * fps)))
    if end_frame <= start_frame:
        raise ValueError('Invalid temporal range.')
    if end_frame - start_frame < num_frames:   # very short last window: repeat frames so the model always gets 8
        idx = np.linspace(start_frame, end_frame - 1, num_frames).round().astype(int).clip(0, n - 1).tolist()
        return torch.from_numpy(vr.get_batch(idx).asnumpy()).permute(0, 3, 1, 2).contiguous()
    k = num_frames
    edges = np.linspace(start_frame, end_frame, k + 1).astype(int)
    idx = [min(int(edges[i]), n - 1) if int(edges[i + 1]) <= int(edges[i]) else (int(edges[i]) + int(edges[i + 1]) - 1) // 2 for i in range(k)]
    return torch.from_numpy(vr.get_batch(idx).asnumpy()).permute(0, 3, 1, 2).contiguous()   # T,C,H,W uint8

def window_frame_indices(n, fps, start_sec, end_sec, num_frames=8):
    start_frame = max(0, int(round(start_sec * fps))); end_frame = min(n, int(round(end_sec * fps)))
    if end_frame <= start_frame: end_frame = min(n, start_frame + 1)
    if end_frame - start_frame < num_frames:
        return np.linspace(start_frame, end_frame - 1, num_frames).round().astype(int).clip(0, n - 1).tolist()
    edges = np.linspace(start_frame, end_frame, num_frames + 1).astype(int)
    return [min(int(edges[i]), n - 1) if int(edges[i + 1]) <= int(edges[i]) else (int(edges[i]) + int(edges[i + 1]) - 1) // 2 for i in range(num_frames)]

_model = None
def get_model():
    global _model
    if _model is None:
        from transformers import AutoModel
        _model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True, torch_dtype=DTYPE).eval().to(DEVICE)
        print('InternVideo2 loaded')
    return _model

V_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3, 1, 1)
V_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3, 1, 1)

def preprocess(frames):
    # official InternVideo2 demo preprocessing: resize to 224x224, scale to [0,1], ImageNet mean/std, shape [1,T,3,224,224]
    x = frames.float() / 255.0                                            # T,C,H,W
    x = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
    return ((x.unsqueeze(0) - V_MEAN) / V_STD)

@torch.inference_mode()
def encode_clip(frames):
    feat = get_model().encode_vision(preprocess(frames).to(DEVICE, DTYPE), test=True).float()
    return F.normalize(feat, dim=-1).reshape(-1).cpu()

def l2n(x, axis=-1):
    return x / np.clip(np.linalg.norm(x, axis=axis, keepdims=True), 1e-12, None)

def slug(name):
    return re.sub(r'[^\w\-.]+', '_', Path(name).stem, flags=re.UNICODE).strip('._')[:90] or 'video'

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()

def save_npz_atomic(path, **kw):
    tmp = Path(str(path) + '.tmp.npz'); np.savez_compressed(tmp, **kw); os.replace(tmp, path)

def save_json_atomic(path, obj):
    tmp = Path(str(path) + '.tmp'); tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=1), encoding='utf-8'); os.replace(tmp, path)
import imageio_ffmpeg
FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()   # ffmpeg shipped via pip (no conda needed)
print('ffmpeg:', FFMPEG)
print('functions ready')

In [ ]:
# 5) Index the old v1.0 checkpoints by video filename (so the 100 already-extracted MVs are reused, not recomputed)
OLD_INDEX = {}
if OLD_CKPT.exists():
    for p in OLD_CKPT.glob('*.npz'):
        try:
            z = np.load(p, allow_pickle=False)
            if 'clip_embeddings' in z and z['clip_embeddings'].ndim == 2 and z['clip_embeddings'].shape[1] == EMB_DIM:
                OLD_INDEX[Path(str(z['relative_path'].item())).name] = p
        except Exception as e:
            print('unreadable old checkpoint', p.name, e)
print('old checkpoints with window embeddings:', len(OLD_INDEX))

# video-level vectors of the FIRST REAL RUN (files in .../embeddings, identified by the video's SHA-256)
OLD_VEC = {}
if OLD_ANALYSIS is not None and (OLD_ANALYSIS / 'embeddings').exists():
    for p in (OLD_ANALYSIS / 'embeddings').rglob('*.npz'):
        try:
            z = np.load(p, allow_pickle=False)
            OLD_VEC[str(z['source_sha256'].item())] = dict(vec=z['embedding'].astype(np.float64), n_windows=int(z['n_windows']),
                                                           duration=float(z['duration_sec']))
        except Exception as e:
            print('skip', p.name, e)
print('first-run video vectors found:', len(OLD_VEC))

In [ ]:
# 6) One pass per video: every stage is saved separately in OUT/per_video/<video>/ and skipped next time
YUNET = OUT / 'face_detection_yunet_2023mar.onnx'
if not YUNET.exists():
    try:
        urllib.request.urlretrieve('https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx', YUNET)
    except Exception as e:
        print('YuNet download failed (Haar fallback will be used):', e)

def face_detector():
    if YUNET.exists():
        det = cv2.FaceDetectorYN.create(str(YUNET), '', (320, 320), 0.6)
        def f(img):
            det.setInputSize((img.shape[1], img.shape[0])); _, faces = det.detect(img)
            return [] if faces is None else [tuple(map(float, fc[:4])) for fc in faces]
        return f, 'yunet'
    haar = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    return (lambda img: [tuple(map(float, b)) for b in haar.detectMultiScale(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), 1.1, 5)]), 'haar'

def colorfulness(img):
    b, g, r = [c.astype(np.float32) for c in cv2.split(img)]
    rg, yb = r - g, 0.5 * (r + g) - b
    return float(np.sqrt(rg.std() ** 2 + yb.std() ** 2) + 0.3 * np.sqrt(rg.mean() ** 2 + yb.mean() ** 2))

def stage_embeddings(path, d, force_new=False):
    out = d / 'embeddings.npz'
    if out.exists() and not force_new: return 'skip'
    if not force_new and path.name in OLD_INDEX:                     # reuse frozen v1.0 window embeddings
        z = np.load(OLD_INDEX[path.name], allow_pickle=False)
        save_npz_atomic(out, clip_embeddings=z['clip_embeddings'], clip_starts=z['clip_starts'], clip_ends=z['clip_ends'],
                        fps=z['fps'], duration_sec=z['duration_sec'], source='v1.0 checkpoint ' + OLD_INDEX[path.name].name)
        return 'reused v1.0'
    vr = VideoReader(str(path), ctx=cpu(0)); fps = float(vr.get_avg_fps()); n = len(vr); dur = n / fps; del vr; gc.collect()
    wins = [(s_, min(dur, s_ + CLIP_SECONDS)) for s_ in make_clip_starts(dur)]
    win_idx = [window_frame_indices(n, fps, s_, e_, NUM_FRAMES) for s_, e_ in wins]
    need = {}                                      # frame index -> windows that use it
    for w, idx in enumerate(win_idx):
        for j in idx: need.setdefault(j, set()).add(w)
    # memory-light: read the video ONCE sequentially (no decord batch buffers), keep only frames still needed
    cap = cv2.VideoCapture(str(path)); buf = {}; E = [None] * len(wins); nxt = 0; k = 0
    last_needed = max(need)
    while k <= last_needed:
        if k in need:
            ok, fr = cap.read()
            if not ok: break
            buf[k] = torch.from_numpy(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).contiguous()
        else:
            if not cap.grab(): break
        while nxt < len(wins) and all(j in buf for j in win_idx[nxt]):     # window complete -> encode, free frames
            E[nxt] = encode_clip(torch.stack([buf[j] for j in win_idx[nxt]])).numpy().astype(np.float32)
            for j in set(win_idx[nxt]):
                need[j].discard(nxt)
                if not need[j]: buf.pop(j, None)
            nxt += 1
        k += 1
    cap.release()
    while nxt < len(wins):                         # frames past the decodable end: repeat the last decoded frame
        last = buf[max(buf)] if buf else None
        E[nxt] = encode_clip(torch.stack([buf.get(j, last) for j in win_idx[nxt]])).numpy().astype(np.float32); nxt += 1
    save_npz_atomic(out, clip_embeddings=np.stack(E), clip_starts=np.array([w[0] for w in wins], np.float32),
                    clip_ends=np.array([w[1] for w in wins], np.float32), fps=np.float32(fps), duration_sec=np.float32(dur),
                    source='extracted v2.0 (frozen v1.0 settings, sequential cv2 read)')
    buf.clear(); gc.collect()
    return 'extracted'

def stage_visual(path, d, motion_fps=10.0, face_fps=2.0, small_w=320):
    out = d / 'visual.npz'
    if out.exists(): return 'skip'
    cap = cv2.VideoCapture(str(path)); fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    m_step, f_step = max(1, round(fps / motion_fps)), max(1, round(fps / face_fps))
    det, det_name = face_detector()
    t_m, motion, cam = [], [], []
    t_f, lum, con, col, nface, maxface, face_off = [], [], [], [], [], [], []
    prev = None; k = 0; hann = None
    while True:
        ok, fr = cap.read()
        if not ok: break
        if k % m_step == 0:
            small = cv2.resize(fr, (small_w, int(fr.shape[0] * small_w / fr.shape[1])))
            g = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
            if prev is not None:
                if hann is None: hann = cv2.createHanningWindow(g.shape[::-1], cv2.CV_32F)
                (dx, dy), _ = cv2.phaseCorrelate(prev, g, hann)           # global shift ~ camera pan/tilt
                t_m.append(k / fps); motion.append(float(np.abs(g - prev).mean())); cam.append(float(np.hypot(dx, dy) / small_w))
            prev = g
        if k % f_step == 0:
            small = cv2.resize(fr, (small_w, int(fr.shape[0] * small_w / fr.shape[1])))
            g = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
            t_f.append(k / fps); lum.append(float(g.mean())); con.append(float(g.std())); col.append(colorfulness(small))
            faces = det(fr); H, W = fr.shape[:2]; nface.append(len(faces))
            if faces:
                areas = [a[2] * a[3] / (H * W) for a in faces]; j = int(np.argmax(areas)); x, y, w, h = faces[j]
                maxface.append(areas[j]); face_off.append(float(np.hypot((x + w / 2) / W - 0.5, (y + h / 2) / H - 0.5)))
            else:
                maxface.append(0.0); face_off.append(np.nan)
        k += 1
    cap.release()
    save_npz_atomic(out, t_motion=np.array(t_m, np.float32), motion=np.array(motion, np.float32), camera_shift=np.array(cam, np.float32),
                    t_frame=np.array(t_f, np.float32), luminance=np.array(lum, np.float32), contrast=np.array(con, np.float32),
                    colorfulness=np.array(col, np.float32), n_faces=np.array(nface, np.int16), max_face_area=np.array(maxface, np.float32),
                    face_centre_offset=np.array(face_off, np.float32), fps=np.float32(fps), n_frames=np.int64(k), face_detector=det_name)
    return 'done'

def stage_shots(path, d):
    out = d / 'shots.json'
    if out.exists(): return 'skip'
    from scenedetect import open_video as sd_open, SceneManager
    from scenedetect.detectors import ContentDetector
    sm = SceneManager(); sm.add_detector(ContentDetector(threshold=27.0)); v = sd_open(str(path)); sm.detect_scenes(v)
    shots = [[s.get_seconds(), e.get_seconds()] for s, e in sm.get_scene_list()]
    save_json_atomic(out, {'detector': 'PySceneDetect ContentDetector threshold=27', 'shots': shots})
    return 'done'

def stage_audio(path, d, sr=22050, hop_sec=0.5):
    out = d / 'audio.npz'
    if out.exists(): return 'skip'
    import librosa
    wav = d / 'audio.wav'
    if not wav.exists():
        subprocess.run([FFMPEG, '-loglevel', 'error', '-y', '-i', str(path), '-ac', '1', '-ar', str(sr), str(wav)], check=True)
    y, _ = librosa.load(wav, sr=sr); hop = int(sr * hop_sec); gc.collect()
    tempo, beats = librosa.beat.beat_track(y=y, sr=sr)
    save_npz_atomic(out, hop_sec=np.float32(hop_sec), sr=np.int32(sr),
                    chroma=librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=hop).astype(np.float32),
                    mfcc=librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20, hop_length=hop).astype(np.float32),
                    rms=librosa.feature.rms(y=y, hop_length=hop)[0].astype(np.float32),
                    onset=librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop).astype(np.float32),
                    tempo=np.float32(np.atleast_1d(tempo)[0]), beat_times=librosa.frames_to_time(beats, sr=sr).astype(np.float32))
    return 'done'

def stage_contact(path, d, seg=8.0, cols=6, tw=320):
    out = d / 'contact_sheet.jpg'
    if out.exists(): return 'skip'
    from PIL import Image, ImageDraw
    cap = cv2.VideoCapture(str(path)); fps = cap.get(cv2.CAP_PROP_FPS) or 25.0; n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); dur = n / fps
    times = np.arange(seg / 2, dur, seg); want = {min(n - 1, int(t * fps)): i for i, t in enumerate(times)}; got = {}; k = 0
    while k <= max(want):
        if k in want:
            ok, f_ = cap.read()
            if not ok: break
            got[want[k]] = cv2.cvtColor(f_, cv2.COLOR_BGR2RGB)
        elif not cap.grab(): break
        k += 1
    cap.release(); fr = np.stack([got[i] for i in sorted(got)]); times = times[sorted(got)]
    th = int(fr.shape[1] * tw / fr.shape[2]); rows = math.ceil(len(fr) / cols)
    sheet = Image.new('RGB', (cols * tw, rows * (th + 18)), 'white'); dr = ImageDraw.Draw(sheet)
    for i, (f, t) in enumerate(zip(fr, times)):
        x, y = (i % cols) * tw, (i // cols) * (th + 18); s0 = int(t - seg / 2)
        sheet.paste(Image.fromarray(f).resize((tw, th)), (x, y + 18)); dr.text((x + 4, y + 2), f'seg {i+1:02d}  {s0//60}:{s0%60:02d}', fill='black')
    tmp = d / 'contact_sheet.tmp.jpg'; sheet.save(tmp, quality=80); os.replace(tmp, out)
    del fr, sheet, got; gc.collect()
    return 'done'

STAGES = [('embeddings', stage_embeddings), ('visual', stage_visual), ('shots', stage_shots), ('audio', stage_audio), ('contact', stage_contact)]

def process_corpus(src_dir, corpus, only=None):
    base = OUT / 'per_video' / corpus; base.mkdir(parents=True, exist_ok=True)
    log_path = OUT / 'tables' / f'run_log_{corpus}.csv'
    files = list_videos(src_dir)[: only or None]
    for i, p in enumerate(files):
        d = base / slug(p.name); d.mkdir(exist_ok=True)
        info = d / 'info.json'
        if not info.exists():
            save_json_atomic(info, {'file_name': p.name, 'relative_path': str(p.relative_to(src_dir)), 'sha256': sha256(p), 'size_mb': p.stat().st_size / 2**20})
        status = {}
        for name, fn in STAGES:
            try:
                t0 = time.time(); status[name] = fn(p, d); status[name + '_s'] = round(time.time() - t0, 1)
            except Exception as e:
                status[name] = 'ERROR: ' + repr(e)[:200]
        row = pd.DataFrame([{'time': time.strftime('%Y-%m-%d %H:%M'), 'file_name': p.name, **status}])
        row.to_csv(log_path, mode='a', header=not log_path.exists(), index=False)
        gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()
        import psutil; ram = psutil.virtual_memory().percent
        print(f'[{i+1}/{len(files)}] RAM {ram:.0f}% | {p.name[:55]} ->', {k: v for k, v in status.items() if not k.endswith('_s')})
        if ram > 85:
            print('RAM above 85%: stopping safely. Restart the session (Runtime > Restart session), run cells 1-6, then cell 8 again; finished videos are skipped.')
            return
print('pipeline ready')

In [ ]:
# 7) SMOKE TEST (first time only): extract ONE video from scratch and compare with the old v1.0 window embeddings
smoke = list_videos(KPOP_DIR)[0]
d = OUT / 'smoke_test' / slug(smoke.name); d.mkdir(parents=True, exist_ok=True)
for f in d.glob('*'): f.unlink()
print('smoke video:', smoke.name)
for name, fn in STAGES:
    t0 = time.time(); r = fn(smoke, d, force_new=True) if name == 'embeddings' else fn(smoke, d)
    print(f'  {name:10s}: {r} ({time.time()-t0:.0f}s)')
z = np.load(d / 'embeddings.npz'); E = z['clip_embeddings']
v = np.load(d / 'visual.npz'); a = np.load(d / 'audio.npz'); sh = json.loads((d / 'shots.json').read_text(encoding='utf-8'))
checks = {
    'embedding shape ok': E.ndim == 2 and E.shape[1] == EMB_DIM,
    'embeddings finite & unit length': bool(np.isfinite(E).all() and np.allclose(np.linalg.norm(E, axis=1), 1, atol=1e-3)),
    'visual series non-empty': len(v['luminance']) > 10 and len(v['motion']) > 10,
    'faces detected somewhere': int(v['n_faces'].sum()) > 0,
    'shots detected': len(sh['shots']) > 1,
    'audio features ok': a['chroma'].shape[0] == 12 and a['chroma'].shape[1] > 50,
    'contact sheet saved': (d / 'contact_sheet.jpg').exists(),
}
if smoke.name in OLD_INDEX:
    old = np.load(OLD_INDEX[smoke.name])['clip_embeddings']
    same_n = old.shape == E.shape
    cos_min = float((l2n(old) * l2n(E)).sum(1).min()) if same_n else float('nan')
    checks['same number of windows as v1.0'] = same_n
    checks[f'window vectors match v1.0 (min cos={cos_min:.4f} > 0.995)'] = same_n and cos_min > 0.995
ov = OLD_VEC.get(json.loads((d / 'info.json').read_text(encoding='utf-8'))['sha256'] if (d / 'info.json').exists() else sha256(smoke))
if ov is not None:
    c = float(l2n(ov['vec']) @ l2n(E.mean(0)))
    print(f'cosine with first-run vector = {c:.4f}  (>0.995 identical pipeline; 0.98-0.995 nearly identical, tell me; <0.98 different)')
    checks['video vector matches first run (cos > 0.995)'] = c > 0.995
    if ov['n_windows'] is not None:
        checks[f"window count matches first run ({len(E)} vs {ov['n_windows']})"] = len(E) == ov['n_windows']
else:
    print('NOTE: no first-run vector found for this video; reproducibility vs. the first run is NOT checked.')
for k, ok in checks.items(): print(('OK   ' if ok else 'FAIL ') + k)
print('\nSMOKE TEST PASSED' if all(checks.values()) else '\nSMOKE TEST FAILED -> send me this output')

In [ ]:
# 8) MAIN RUN: all K-pop videos (step 1). Safe to re-run after a disconnect: finished stages are skipped.
process_corpus(KPOP_DIR, 'kpop')
log = pd.read_csv(OUT / 'tables' / 'run_log_kpop.csv')
errs = log[log.astype(str).apply(lambda r: r.str.startswith('ERROR').any(), axis=1)]
print('\nvideos with an ERROR in the latest pass:', errs['file_name'].nunique(), '(re-run this cell; if it repeats, send me the error)')

In [ ]:
# 9) ANALYSIS of the K-pop corpus -> OUT/tables/kpop_product_analysis.xlsx (+ figures)
from scipy.stats import spearmanr, mannwhitneyu
import matplotlib.pyplot as plt

def load_corpus(corpus):
    rows = {}
    for d in sorted((OUT / 'per_video' / corpus).iterdir()):
        need = ['info.json', 'embeddings.npz', 'visual.npz', 'shots.json', 'audio.npz']
        if not all((d / n).exists() for n in need): continue
        info = json.loads((d / 'info.json').read_text(encoding='utf-8')); e = np.load(d / 'embeddings.npz', allow_pickle=True)
        W = l2n(e['clip_embeddings'].astype(np.float64))
        rows[info['file_name']] = dict(dir=d, W=W, starts=e['clip_starts'], ends=e['clip_ends'], dur=float(e['duration_sec']),
                                       v=l2n(W.mean(0)), u=W.mean(0))
    return rows

K = load_corpus('kpop')
meta = pd.read_excel(METADATA, sheet_name='Metadata_105')
import unicodedata
def _key(x): return re.sub(r'[^0-9a-z\uac00-\ud7a3]+', '', unicodedata.normalize('NFC', str(x)).lower())
_kmap = {_key(k): k for k in K}
meta['file_name'] = [ _kmap.get(_key(x), x) for x in meta['file_name'] ]     # match names robustly (unicode / quotes / spaces)
unmatched_meta = [x for x in meta['file_name'] if x not in K]
print('metadata rows not matched to a processed video:', unmatched_meta)
meta = meta[meta['file_name'].isin(K)].reset_index(drop=True)
missing = sorted(set(K) - set(meta['file_name']))
print('videos analysed:', len(meta), '| in Drive but not in metadata:', missing)
names = list(meta['file_name']); n = len(names)
fem = (meta['gender'].str.lower() == 'female').to_numpy()
V = np.stack([K[x]['v'] for x in names]); S = V @ V.T; np.fill_diagonal(S, np.nan)
zs = lambda x: (x - np.nanmean(x)) / np.nanstd(x, ddof=1)

def balanced(Smat, f):
    return 0.5 * np.nanmean(np.where(f[None, :], Smat, np.nan), 1) + 0.5 * np.nanmean(np.where(~f[None, :], Smat, np.nan), 1)

T = meta[['file_name', 'artist', 'song', 'gender', 'release_year', 'mv_release_date', 'generation_by_debut_rule']].copy()
T['index_raw'] = np.nanmean(S, 1); T['index_raw_z'] = zs(T['index_raw'])
T['index_balanced'] = balanced(S, fem); T['index_balanced_z'] = zs(T['index_balanced'])          # PRIMARY INDEX
within = np.where(fem, np.nanmean(np.where(fem[None, :], S, np.nan), 1), np.nanmean(np.where(~fem[None, :], S, np.nan), 1))
T['index_within_gender_z'] = np.nan
T.loc[fem, 'index_within_gender_z'] = zs(within[fem]); T.loc[~fem, 'index_within_gender_z'] = zs(within[~fem])
T['index_knn10_z'] = zs(np.array([np.sort(r[~np.isnan(r)])[-10:].mean() for r in S]))
U = np.stack([K[x]['u'] for x in names]); Su = U @ U.T; np.fill_diagonal(Su, np.nan)
T['index_windowlevel_z'] = zs(balanced(Su, fem))

# within-video structure: visual repetition (non-adjacent windows), resultant length, n windows
rep, R, nw = [], [], []
for x in names:
    W = K[x]['W']; G = W @ W.T; st = K[x]['starts']
    far = np.abs(np.subtract.outer(st, st)) >= CLIP_SECONDS          # windows that do not overlap
    rep.append(G[far].mean() if far.any() else np.nan); R.append(np.linalg.norm(W.mean(0))); nw.append(len(W))
T['visual_repetition'] = rep; T['resultant_R'] = R; T['n_windows'] = nw

# automatic features (means over the whole video)
def feats(d):
    v = np.load(d / 'visual.npz'); sh = json.loads((d / 'shots.json').read_text(encoding='utf-8'))['shots']; a = np.load(d / 'audio.npz')
    dur = float(v['n_frames']) / float(v['fps']); L = [e - s for s, e in sh] or [dur]
    mf = v['max_face_area']
    return dict(luminance=v['luminance'].mean(), contrast=v['contrast'].mean(), colorfulness=v['colorfulness'].mean(),
                motion=v['motion'].mean(), camera_movement=v['camera_shift'].mean(), cuts_per_min=60 * max(len(sh) - 1, 0) / dur,
                median_shot_s=float(np.median(L)), faces_per_frame=v['n_faces'].mean(), closeup_share=float((mf >= 0.05).mean()),
                face_centre_offset=np.nanmean(v['face_centre_offset']), tempo_bpm=float(a['tempo']))
FEAT = pd.DataFrame([dict(file_name=x, **feats(K[x]['dir'])) for x in names])

# audio repetition + chorus candidates from saved features (no video reading)
def audio_structure(d, seg_sec=10.0, min_start=15.0):
    a = np.load(d / 'audio.npz'); hop = float(a['hop_sec']); m = a['mfcc'][:13]
    Fm = np.vstack([a['chroma'], (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-9)]).T
    Fm = l2n(Fm); Sm = Fm @ Fm.T; nn = len(Fm); L = int(seg_sec / hop)
    seg = np.full((nn - L, nn - L), np.nan)
    for lag in range(L, nn - L):
        c = np.convolve(np.diagonal(Sm, lag), np.ones(L) / L, mode='valid'); mm = min(len(c), nn - L - lag)
        seg[np.arange(mm), np.arange(mm) + lag] = c[:mm]; seg[np.arange(mm) + lag, np.arange(mm)] = c[:mm]
    reps = np.nansum(seg > np.nanpercentile(seg, 90), 1)
    rms = a['rms'][: nn - L]; score = reps / (reps.max() + 1e-9) + 0.25 * (rms - rms.mean()) / (rms.std() + 1e-9)
    t = np.arange(len(score)) * hop; cands = []
    for i in np.argsort(-score):
        if t[i] >= min_start and all(abs(t[i] - c) >= 20 for c in cands): cands.append(float(t[i]))
        if len(cands) == 3: break
    return dict(audio_repetition=float(np.mean(reps > 0)), chorus_cand_1=cands[0] if cands else np.nan,
                chorus_cand_2=cands[1] if len(cands) > 1 else np.nan, chorus_cand_3=cands[2] if len(cands) > 2 else np.nan)
AUD = pd.DataFrame([dict(file_name=x, **audio_structure(K[x]['dir'])) for x in names])
T = T.merge(FEAT, on='file_name').merge(AUD, on='file_name')

# --- reports ---
summary = []
summary.append(('n videos', n)); summary.append(('mean pairwise cosine', np.nanmean(S)))
summary.append(('norm of mean vector (squared ~ mean cosine)', np.linalg.norm(V.mean(0))))
for c in ['visual_repetition', 'resultant_R', 'n_windows', 'index_windowlevel_z', 'index_raw_z', 'index_within_gender_z', 'index_knn10_z']:
    r, p = spearmanr(T[c], T['index_balanced_z'], nan_policy='omit'); summary.append((f'rho({c}, balanced index)', f'{r:+.3f} (p={p:.3f})'))
Sf, Sm_, Sx = S[np.ix_(fem, fem)], S[np.ix_(~fem, ~fem)], S[np.ix_(fem, ~fem)]
summary += [('within-female similarity', np.nanmean(Sf)), ('within-male similarity', np.nanmean(Sm_)), ('between-gender similarity', np.nanmean(Sx))]
d_raw = (T.loc[fem, 'index_raw'].mean() - T.loc[~fem, 'index_raw'].mean()) / T['index_raw'].std()
d_bal = (T.loc[fem, 'index_balanced'].mean() - T.loc[~fem, 'index_balanced'].mean()) / T['index_balanced'].std()
summary += [('gender gap, raw index (d)', d_raw), ('gender gap, balanced index (d)', d_bal)]
summary.append(('averaging check rule', 'OK if |rho(visual_repetition)| < .30; if rho <= -.30 report index_windowlevel too'))
SUM = pd.DataFrame(summary, columns=['quantity', 'value'])

# gender types: standardised F-M differences with act-level cluster bootstrap
rng = np.random.default_rng(2026); acts = T['artist'].unique(); GT = []
for c in ['luminance', 'contrast', 'colorfulness', 'motion', 'camera_movement', 'cuts_per_min', 'median_shot_s', 'faces_per_frame',
          'closeup_share', 'face_centre_offset', 'tempo_bpm', 'audio_repetition', 'visual_repetition']:
    sd = T[c].std(); obs = (T.loc[fem, c].mean() - T.loc[~fem, c].mean()) / sd; bs = []
    for _ in range(2000):
        b = pd.concat([T[T.artist == a_] for a_ in rng.choice(acts, len(acts))]); bf = b[b.gender.str.lower() == 'female'][c]; bm = b[b.gender.str.lower() == 'male'][c]
        if len(bf) and len(bm): bs.append((bf.mean() - bm.mean()) / sd)
    GT.append(dict(feature=c, female_mean=T.loc[fem, c].mean(), male_mean=T.loc[~fem, c].mean(), d_F_minus_M=obs,
                   ci_low=np.nanpercentile(bs, 2.5), ci_high=np.nanpercentile(bs, 97.5)))
GT = pd.DataFrame(GT)

# which formal features go with the index? (exploratory path analysis)
PATH = pd.DataFrame([dict(feature=c, rho_with_balanced_index=spearmanr(T[c], T['index_balanced_z'], nan_policy='omit')[0])
                     for c in GT['feature']])

# time trend (exploratory): mixed model with random intercept for act
TT = T.copy(); TT['year_frac'] = pd.to_datetime(TT['mv_release_date']).dt.year + pd.to_datetime(TT['mv_release_date']).dt.dayofyear / 365.25
try:
    import statsmodels.formula.api as smf
    mm = smf.mixedlm('index_balanced_z ~ year_frac + C(gender)', TT, groups=TT['artist']).fit()
    TREND = pd.DataFrame({'term': mm.params.index, 'coef': mm.params.values, 'se': mm.bse.values, 'p': mm.pvalues.values})
except Exception as e:
    TREND = pd.DataFrame({'note': [f'mixed model failed: {e}']})

# stratified sample of 30 for human coding (tertiles x gender, max 2 per act), blind list + segment coding sheet
rng = np.random.default_rng(2026)
T['tertile'] = pd.qcut(T['index_balanced_z'], 3, labels=['low', 'mid', 'high'])
picked = []
for ter in ['low', 'mid', 'high']:
    for g, k in [('female', 5), ('male', 5)]:
        pool = T[(T.tertile == ter) & (T.gender.str.lower() == g)].sample(frac=1, random_state=int(rng.integers(1e9)))
        for _, r in pool.iterrows():
            if sum(1 for p_ in picked if p_['artist'] == r['artist']) < 2 and len([p_ for p_ in picked if p_['tertile'] == ter and p_['gender'].lower() == g]) < k:
                picked.append(r.to_dict())
SAMPLE = pd.DataFrame(picked)
BLIND = SAMPLE[['file_name']].sample(frac=1, random_state=7).reset_index(drop=True)
BLIND.insert(0, 'code_id', [f'V{i+1:02d}' for i in range(len(BLIND))])
seg_rows = []
for _, r in BLIND.iterrows():
    dur = K[r['file_name']]['dur']
    for sidx, s0 in enumerate(np.arange(0, dur, 8.0)):
        seg_rows.append(dict(coder='', code_id=r['code_id'], file_name=r['file_name'], segment_no=sidx + 1, start_s=int(s0),
                             is_chorus='', shot='', performance='', setting='', n_performers='', camera_gaze='', point_move='', note=''))
for c_ in ['contact_sheets_for_coding']:
    (OUT / 'tables' / c_).mkdir(exist_ok=True)
for _, r in BLIND.iterrows():
    shutil.copy(K[r['file_name']]['dir'] / 'contact_sheet.jpg', OUT / 'tables' / 'contact_sheets_for_coding' / f"{r['code_id']}.jpg")

CHORUS = T[['file_name', 'chorus_cand_1', 'chorus_cand_2', 'chorus_cand_3']].copy()
CHORUS['chorus_start_verified'] = np.nan; CHORUS['exclude_for_eyetracking'] = ''; CHORUS['note'] = ''
chorus_path = OUT / 'tables' / 'chorus_check.xlsx'
if not chorus_path.exists(): CHORUS.to_excel(chorus_path, index=False)

with pd.ExcelWriter(OUT / 'tables' / 'kpop_product_analysis.xlsx', engine='xlsxwriter') as xw:
    SUM.to_excel(xw, sheet_name='summary', index=False); T.drop(columns=['tertile']).to_excel(xw, sheet_name='per_video', index=False)
    GT.to_excel(xw, sheet_name='gender_types', index=False); PATH.to_excel(xw, sheet_name='index_vs_features', index=False)
    TREND.to_excel(xw, sheet_name='time_trend', index=False)
    SAMPLE[['file_name', 'artist', 'gender', 'tertile', 'index_balanced_z']].to_excel(xw, sheet_name='coding_sample_KEY', index=False)
with pd.ExcelWriter(OUT / 'tables' / 'coding_sheet_BLIND.xlsx', engine='xlsxwriter') as xw:
    BLIND.to_excel(xw, sheet_name='videos', index=False); pd.DataFrame(seg_rows).drop(columns=['file_name']).to_excel(xw, sheet_name='Segment_coding', index=False)
    pd.DataFrame({'code_id': BLIND['code_id'], 'coder': '', 'outfits': '', 'sets': '', 'concept': '', 'gender_display': '', 'memo': ''}).to_excel(xw, sheet_name='Video_level', index=False)
pd.DataFrame(S, index=names, columns=names).to_csv(OUT / 'tables' / 'similarity_matrix_kpop.csv', encoding='utf-8-sig')

# figures
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(T['index_balanced_z'], bins=20); ax[0].set_title('balanced index (z)')
for g, c in [('Female', 'tab:red'), ('Male', 'tab:blue')]:
    sel = T.gender == g; ax[1].scatter(T.loc[sel, 'index_raw_z'], T.loc[sel, 'index_balanced_z'], c=c, label=g, s=14)
ax[1].set_xlabel('raw index (z)'); ax[1].set_ylabel('balanced index (z)'); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT / 'figures' / 'index_distribution.png', dpi=150); plt.show()
display(SUM); display(GT.round(3))
print('saved: tables/kpop_product_analysis.xlsx, tables/coding_sheet_BLIND.xlsx, tables/contact_sheets_for_coding/, tables/chorus_check.xlsx')

### کارهای انسانی پس از سلول ۹
- برگه‌ی کدگذاری کور (`tables/coding_sheet_BLIND.xlsx`، ویدئوها با کد V01 تا V30، بدون شاخص) و برگه‌های تماس (`tables/contact_sheets_for_coding/`) به کدگذاران داده می‌شود؛ فایل `coding_sample_KEY` داده نمی‌شود. کدگذاری با `coding_coder1.xlsx` و `coding_coder2.xlsx` و راهنمای کدگذاری نسخه‌ی ۳ انجام می‌شود.
- زمان شروع نخستین کورس با شنیدن در `chorus_shortlist.xlsx` (ستون `chorus_start_verified`) وارسی می‌شود؛ ستون‌های `chorus_cand` پیشنهاد ماشین‌اند.


In [ ]:
# 10) Step 3: Korean indie reference corpus -> same visual pipeline, then visual comparison with K-pop
#     Heavy the first time (all stages for every indie video, ~ as slow per video as K-pop). Safe to re-run: finished stages are skipped.
print('indie folder:', INDIE_DIR, '| exists:', INDIE_DIR.exists(), '| videos:', len(list_videos(INDIE_DIR)) if INDIE_DIR.exists() else 0)
process_corpus(INDIE_DIR, 'kindie')
I = load_corpus('kindie'); inames = sorted(I)
print('indie videos with all stages done:', len(inames))

def indie_artist(x):
    s = unicodedata.normalize('NFC', x).lower()
    for key, art in [('lang lee', 'Lang Lee'), ('이랑', 'Lang Lee'), ('adoy', 'ADOY'), ('선우정아', 'swja'), ('swja', 'swja'), ('sunwoojunga', 'swja')]:
        if key in s: return art
    return re.sub(r'_720p.*$', '', x)                      # every other file is its own artist
IART = pd.Series({x: indie_artist(x) for x in inames})
print('indie artists:', IART.nunique(), '| artists with >1 video:', IART.value_counts()[lambda v: v > 1].to_dict())

VI = np.stack([I[x]['v'] for x in inames])
coh = lambda X: (X @ X.T)[np.triu_indices(len(X), 1)].mean()
rng = np.random.default_rng(11)
by_act = meta.groupby('artist')['file_name'].apply(list); by_iart = IART.groupby(IART).apply(lambda s: list(s.index))
k = min(len(by_iart), len(by_act))
one_per = lambda groups, R: np.stack([R[rng.choice(g)]['v'] for g in groups])
kk = np.array([coh(one_per([by_act[a_] for a_ in rng.choice(by_act.index, k, replace=False)], K)) for _ in range(2000)])
ii = np.array([coh(one_per([by_iart[a_] for a_ in rng.choice(by_iart.index, k, replace=False)], I)) for _ in range(2000)])
cross = VI @ V.T
# how K-pop-like is each indie video? (balanced similarity to K-pop, same scale as the K-pop raw index)
ind_bal = 0.5 * cross[:, fem].mean(1) + 0.5 * cross[:, ~fem].mean(1)
# separability (Nie's 'distinctive' convention): leave-one-out nearest-centroid with cosine, K-pop vs indie
def sep_acc(X, lab):
    # leave-one-out classification by class-standardised mean similarity (fair when one class is more diffuse)
    G = X @ X.T; np.fill_diagonal(G, np.nan); hit = np.zeros(len(X), bool); cls = np.unique(lab)
    for j in range(len(X)):
        sc = []
        for c in cls:
            m = (lab == c); m[j] = False
            W_ = G[np.ix_(m, m)]; mu, sd = np.nanmean(W_), np.nanstd(W_) + 1e-9
            sc.append((np.nanmean(G[j, m]) - mu) / sd)
        hit[j] = cls[int(np.argmax(sc))] == lab[j]
    return np.array([hit[lab == c].mean() for c in cls])
def sep_test(X, lab, n_perm=100, seed=3):
    acc = sep_acc(X, lab); bal = acc.mean(); r_ = np.random.default_rng(seed)
    null = np.array([sep_acc(X, r_.permutation(lab)).mean() for _ in range(n_perm)])
    return acc, bal, float(np.mean(null >= bal)), float(np.mean(null))
(acc_k, acc_i), bal_ki, p_ki, null_ki = sep_test(np.vstack([V, VI]), np.r_[np.zeros(len(V)), np.ones(len(VI))])
REF = pd.DataFrame([
    ('indie videos / artists', f'{len(VI)} / {IART.nunique()}'),
    (f'K-pop coherence, {k} videos one per act (median, 95% range)', f'{np.median(kk):.4f} ({np.percentile(kk,2.5):.4f} to {np.percentile(kk,97.5):.4f})'),
    (f'indie coherence, {k} videos one per artist (median, 95% range)', f'{np.median(ii):.4f} ({np.percentile(ii,2.5):.4f} to {np.percentile(ii,97.5):.4f})'),
    ('P(K-pop subset more coherent than indie subset)', f'{np.mean(kk[:, None] > ii[None, :1000]):.3f}'),
    ('mean K-pop x indie similarity', f'{cross.mean():.4f}'),
    ('K-pop raw index (mean similarity to other K-pop), median', f'{np.median(T["index_raw"]):.4f}'),
    ('indie: balanced similarity to K-pop, median', f'{np.median(ind_bal):.4f}'),
    ('share of indie videos more K-pop-like than the median K-pop video', f'{np.mean(ind_bal > np.median(T["index_balanced"])):.3f}'),
    ('separability: leave-one-out accuracy (class-standardised similarity) K-pop / indie', f'{acc_k:.3f} / {acc_i:.3f}; balanced {bal_ki:.3f} vs permutation null {null_ki:.3f}, p={p_ki:.3f}'),
], columns=['quantity', 'value'])

FI = []
for x in inames:
    W = I[x]['W']; st = I[x]['starts']; far = np.abs(np.subtract.outer(st, st)) >= CLIP_SECONDS; G = W @ W.T
    FI.append(dict(file_name=x, artist=IART[x], **feats(I[x]['dir']), audio_repetition=audio_structure(I[x]['dir'])['audio_repetition'],
                   visual_repetition=G[far].mean() if far.any() else np.nan, kpop_likeness=ind_bal[inames.index(x)]))
FI = pd.DataFrame(FI)
CMP = pd.DataFrame([dict(feature=c, kpop_mean=T[c].mean(), indie_mean=FI[c].mean(),
                         d_kpop_minus_indie=(T[c].mean() - FI[c].mean()) / np.sqrt((T[c].var() + FI[c].var()) / 2),
                         p_mannwhitney=mannwhitneyu(T[c].dropna(), FI[c].dropna())[1]) for c in GT['feature']])
o = np.argsort(CMP['p_mannwhitney'].to_numpy()); ph = np.empty(len(o)); run = 0
for i_, j_ in enumerate(o): run = max(run, (len(o) - i_) * CMP['p_mannwhitney'].iloc[j_]); ph[j_] = min(1, run)
CMP['p_holm'] = ph

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
iu = np.triu_indices(len(V), 1); ju = np.triu_indices(len(VI), 1)
ax[0].hist((V @ V.T)[iu], bins=40, alpha=.6, density=True, label='within K-pop')
ax[0].hist((VI @ VI.T)[ju], bins=40, alpha=.6, density=True, label='within indie')
ax[0].hist(cross.ravel(), bins=40, alpha=.6, density=True, label='K-pop x indie'); ax[0].legend(); ax[0].set_title('pairwise cosine similarity')
ax[1].hist(kk, bins=40, alpha=.6, label='K-pop subsets'); ax[1].hist(ii, bins=40, alpha=.6, label='indie subsets'); ax[1].legend()
ax[1].set_title(f'coherence of {k}-video subsets (one per act)')
plt.tight_layout(); plt.savefig(OUT / 'figures' / 'kpop_vs_indie_visual.png', dpi=150); plt.show()
with pd.ExcelWriter(OUT / 'tables' / 'indie_reference_comparison.xlsx', engine='xlsxwriter') as xw:
    REF.to_excel(xw, sheet_name='coherence', index=False); CMP.to_excel(xw, sheet_name='features', index=False)
    FI.to_excel(xw, sheet_name='indie_per_video', index=False)
display(REF); display(CMP.round(3))
print('saved: tables/indie_reference_comparison.xlsx, figures/kpop_vs_indie_visual.png')

In [ ]:
# 11) MIR (music side, Adorno's constructs) + feature typicality + exploratory dimensions
#     Uses only the saved audio.npz / per-video tables (no video reading; a few minutes on CPU).
#     Run after cells 9 and 10. If the indie folder is missing it runs for K-pop only.
from scipy.signal import find_peaks
from scipy.stats import entropy

KS_MAJ = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
KS_MIN = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17])
TRIADS = np.array([np.roll([1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0], r) for r in range(12)] +
                  [np.roll([1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0], r) for r in range(12)], float)

def mir_song(d):
    a = np.load(d / 'audio.npz'); hop = float(a['hop_sec']); C = a['chroma'].astype(float); M = a['mfcc'].astype(float)
    rms = a['rms'].astype(float); on = a['onset'].astype(float); bt = a['beat_times'].astype(float)
    nfr = min(C.shape[1], M.shape[1], len(rms)); C, M, rms = C[:, :nfr], M[:, :nfr], rms[:nfr]; dur = nfr * hop
    # key (Krumhansl-Schmuckler) -> key-invariant chroma profile
    pc = C.mean(1); best = max(((np.corrcoef(pc, np.roll(p, k))[0, 1], k, m) for k in range(12) for m, p in [('maj', KS_MAJ), ('min', KS_MIN)]))
    pc_rel = np.roll(pc / pc.sum(), -best[1])
    # chords (24 triad templates) -> harmonic vocabulary and schematism
    Cn = C / (np.linalg.norm(C, axis=0, keepdims=True) + 1e-9); ch = (TRIADS / np.linalg.norm(TRIADS, axis=1, keepdims=True) @ Cn).argmax(0)
    chg = ch[np.r_[True, ch[1:] != ch[:-1]]]                      # chord sequence without repeats
    cnt = np.bincount(ch, minlength=24) / len(ch)
    trans = [f'{x}>{y}' for x, y in zip(chg[:-1], chg[1:])]; tc = pd.Series(trans).value_counts(normalize=True) if trans else pd.Series([1.0])
    # rhythm and dynamics
    ibi = np.diff(bt); db = 20 * np.log10(rms + 1e-6)
    # form: Foote novelty on chroma+timbre self-similarity -> section boundaries
    Z = np.vstack([Cn, (M[:13] - M[:13].mean(1, keepdims=True)) / (M[:13].std(1, keepdims=True) + 1e-9)]).T
    Z = l2n(Z); SS = Z @ Z.T; w = 16; g = np.outer(np.r_[-np.ones(w), np.ones(w)], np.r_[-np.ones(w), np.ones(w)]) * -1
    nov = np.array([np.sum(SS[i - w:i + w, i - w:i + w] * g) if w <= i < nfr - w else 0 for i in range(nfr)])
    pk, _ = find_peaks(nov, distance=int(6 / hop), prominence=np.std(nov) * 0.5 if nov.std() > 0 else None)
    bounds = np.r_[0, pk, nfr] * hop; secl = np.diff(bounds)
    st = audio_structure(d)
    return dict(
        tempo_bpm=float(a['tempo']), beat_regularity=float(1 - np.std(ibi) / (np.mean(ibi) + 1e-9)) if len(ibi) > 4 else np.nan,
        onset_strength=float(on.mean()), dynamic_range_db=float(np.percentile(db, 95) - np.percentile(db, 10)),
        loudness_variation=float(np.std(db[db > np.percentile(db, 10)])),
        key_clarity=float(best[0]), mode_major=int(best[2] == 'maj'),
        chord_vocabulary=int((cnt > 0.02).sum()), chord_entropy=float(entropy(cnt[cnt > 0], base=2)),
        chord_changes_per_min=float(60 * (len(chg) - 1) / dur), progression_entropy=float(entropy(tc.values, base=2)),
        top4_progressions_share=float(tc.iloc[:4].sum()),
        sections_per_min=float(60 * len(secl) / dur), section_length_regularity=float(1 - np.std(secl) / (np.mean(secl) + 1e-9)),
        audio_repetition=st['audio_repetition'],
        first_hook_s=float(np.nanmin([st['chorus_cand_1'], st['chorus_cand_2'], st['chorus_cand_3']])),
        _timbre=np.r_[M.mean(1), M.std(1)], _harmony=np.r_[pc_rel, cnt], _key=best[1])

MIR_COLS = ['tempo_bpm', 'beat_regularity', 'onset_strength', 'dynamic_range_db', 'loudness_variation', 'key_clarity', 'mode_major',
            'chord_vocabulary', 'chord_entropy', 'chord_changes_per_min', 'progression_entropy', 'top4_progressions_share',
            'sections_per_min', 'section_length_regularity', 'audio_repetition', 'first_hook_s']
CONSTRUCT = {'tempo_bpm': 'rhythmic standardization', 'beat_regularity': 'rhythmic standardization ("beaten time")',
             'onset_strength': 'rhythmic drive', 'dynamic_range_db': 'production standardization (compression)',
             'loudness_variation': 'production standardization (compression)', 'key_clarity': 'harmonic schematism',
             'mode_major': 'harmonic schematism', 'chord_vocabulary': 'harmonic schematism (limited vocabulary)',
             'chord_entropy': 'harmonic schematism', 'chord_changes_per_min': 'harmonic rhythm',
             'progression_entropy': 'formula / interchangeable progressions', 'top4_progressions_share': 'formula / interchangeable progressions',
             'sections_per_min': 'form (part interchangeability)', 'section_length_regularity': 'form (fixed scheme)',
             'audio_repetition': 'repetition / plugging', 'first_hook_s': 'immediacy of the hook (regression of listening)'}

def mir_corpus(R):
    rows, TB, HB = [], [], []
    for x in tqdm(R, desc='MIR'):
        m = mir_song(R[x]['dir']); TB.append(m.pop('_timbre')); HB.append(m.pop('_harmony')); m.pop('_key'); rows.append(dict(file_name=x, **m))
    return pd.DataFrame(rows), np.array(TB), np.array(HB)

MK, TBk, HBk = mir_corpus({x: K[x] for x in names})

# song-to-song AUDIO standardization index (same balanced logic as the visual index), on z-scored descriptors
mu_t, sd_t = TBk.mean(0), TBk.std(0) + 1e-9; mu_h, sd_h = HBk.mean(0), HBk.std(0) + 1e-9
mu_m, sd_m = MK[MIR_COLS].mean(), MK[MIR_COLS].std() + 1e-9
def audio_vec(TB, HB, MM): return l2n(np.hstack([(TB - mu_t) / sd_t, (HB - mu_h) / sd_h, ((MM[MIR_COLS] - mu_m) / sd_m).fillna(0).to_numpy()]))
def part_sim(X): S_ = l2n(X) @ l2n(X).T; np.fill_diagonal(S_, np.nan); return S_
AV = audio_vec(TBk, HBk, MK); SA = AV @ AV.T; np.fill_diagonal(SA, np.nan)
MK['audio_index_balanced_z'] = zs(balanced(SA, fem))
MK['timbre_index_z'] = zs(balanced(part_sim((TBk - mu_t) / sd_t), fem))          # surface / sound
HF = ['chord_vocabulary', 'chord_entropy', 'progression_entropy', 'sections_per_min', 'section_length_regularity', 'audio_repetition']
MK['harmony_form_index_z'] = zs(balanced(part_sim(np.hstack([(HBk - mu_h) / sd_h, ((MK[HF] - mu_m[HF]) / sd_m[HF]).fillna(0).to_numpy()])), fem))  # skeleton
# pseudo-individualization (exploratory): standard skeleton (harmony + form) with a distinctive surface (sound)
MK['pseudo_individualization_z'] = zs(MK['harmony_form_index_z'] - MK['timbre_index_z'])
TM = T[['file_name', 'artist', 'gender', 'index_balanced_z', 'visual_repetition']].merge(MK, on='file_name')

mres = []
for c in ['audio_index_balanced_z', 'harmony_form_index_z', 'timbre_index_z', 'pseudo_individualization_z']:
    r, p = spearmanr(TM[c], TM['index_balanced_z']); mres.append((f'rho({c}, VISUAL index)', f'{r:+.3f} (p={p:.3f})'))
r, p = spearmanr(TM['audio_repetition'], TM['audio_index_balanced_z']); mres.append(('rho(audio_repetition, AUDIO index)', f'{r:+.3f} (p={p:.3f})'))
r, p = spearmanr(TM['audio_repetition'], TM['visual_repetition']); mres.append(('rho(audio_repetition, visual_repetition)', f'{r:+.3f} (p={p:.3f})'))
MSUM = pd.DataFrame(mres, columns=['quantity', 'value'])
MDESC = pd.DataFrame([dict(feature=c, construct=CONSTRUCT[c], kpop_mean=TM[c].mean(), kpop_sd=TM[c].std(),
                           rho_with_audio_index=spearmanr(TM[c], TM['audio_index_balanced_z'], nan_policy='omit')[0],
                           rho_with_visual_index=spearmanr(TM[c], TM['index_balanced_z'], nan_policy='omit')[0]) for c in MIR_COLS])

# TYPICALITY: standardization = being close to the corpus centre, not having a high value
FCOLS = list(GT['feature']); FZ = (T[FCOLS] - T[FCOLS].mean()) / T[FCOLS].std()
TYP = []
for c in FCOLS:
    r, p = spearmanr(-FZ[c].abs(), T['index_balanced_z'], nan_policy='omit'); TYP.append(dict(feature=c, rho_typicality_with_index=r, p=p))
TYP = pd.DataFrame(TYP)
o = np.argsort(TYP['p'].to_numpy()); ph = np.empty(len(o)); run = 0
for i_, j_ in enumerate(o): run = max(run, (len(o) - i_) * TYP['p'].iloc[j_]); ph[j_] = min(1, run)
TYP['p_holm'] = ph                                                                # corrected for 13 tests
X_ = FZ.fillna(0).to_numpy(); maha = np.sqrt(np.einsum('ij,jk,ik->i', X_, np.linalg.pinv(np.cov(X_, rowvar=False)), X_))
r_t, p_t = spearmanr(-maha, T['index_balanced_z'])
TYP = pd.concat([TYP, pd.DataFrame([dict(feature='ALL FEATURES (Mahalanobis typicality)', rho_typicality_with_index=r_t, p=p_t)])], ignore_index=True)

# exploratory dimensions of K-pop form (PCA + varimax on the formal and MIR features)
def varimax(L, it=100, tol=1e-6):
    p_, k_ = L.shape; Rm = np.eye(k_); dd = 0
    for _ in range(it):
        Lr = L @ Rm; u, s, vh = np.linalg.svd(L.T @ (Lr ** 3 - Lr @ np.diag((Lr ** 2).sum(0)) / p_)); Rm = u @ vh
        if s.sum() < dd * (1 + tol): break
        dd = s.sum()
    return L @ Rm
DIMC = FCOLS + [c for c in MIR_COLS if c not in FCOLS and c != 'mode_major']
DZ = TM.merge(T[['file_name'] + [c for c in FCOLS if c not in TM]], on='file_name')[DIMC]
DZ = DZ.loc[:, DZ.std() > 0]; DIMC = list(DZ.columns); DZ = ((DZ - DZ.mean()) / DZ.std()).fillna(0)
ev, evec = np.linalg.eigh(np.corrcoef(DZ.to_numpy(), rowvar=False)); ev, evec = ev[::-1], evec[:, ::-1]
nf = int(max(2, min(6, (ev > 1).sum())))
LOAD = pd.DataFrame(varimax(evec[:, :nf] * np.sqrt(ev[:nf])), index=DIMC, columns=[f'D{i+1}' for i in range(nf)])
SCORES = DZ.to_numpy() @ np.linalg.pinv(LOAD.to_numpy().T)
DIMR = pd.DataFrame([dict(dimension=f'D{i+1}', variance_share=ev[i] / ev.sum(),
                          top_loadings=', '.join(f'{k}({LOAD.loc[k, f"D{i+1}"]:+.2f})' for k in LOAD[f'D{i+1}'].abs().sort_values(ascending=False).index[:4]),
                          rho_level_with_index=spearmanr(SCORES[:, i], TM['index_balanced_z'])[0],
                          rho_typicality_with_index=spearmanr(-np.abs(zs(SCORES[:, i])), TM['index_balanced_z'])[0]) for i in range(nf)])

# K-pop vs indie (only if cell 10 has processed the indie corpus)
CMPM = None
if (OUT / 'per_video' / 'kindie').exists():
    Ii = load_corpus('kindie')
    if len(Ii):
        MI, TBi, HBi = mir_corpus(Ii)
        # pooled standardization (K-pop + indie together), then coherence inside each corpus
        TBp, HBp, MMp = np.vstack([TBk, TBi]), np.vstack([HBk, HBi]), pd.concat([MK[MIR_COLS], MI[MIR_COLS]])
        Zp = l2n(np.hstack([(TBp - TBp.mean(0)) / (TBp.std(0) + 1e-9), (HBp - HBp.mean(0)) / (HBp.std(0) + 1e-9),
                            ((MMp - MMp.mean()) / (MMp.std() + 1e-9)).fillna(0).to_numpy()]))
        Zk, Zi = Zp[:len(MK)], Zp[len(MK):]; coh_ = lambda X: (X @ X.T)[np.triu_indices(len(X), 1)].mean()
        rg = np.random.default_rng(12); kk_ = np.array([coh_(Zk[rg.choice(len(Zk), len(Zi), replace=False)]) for _ in range(2000)])
        CMPM = pd.DataFrame([dict(feature=c, construct=CONSTRUCT[c], kpop_mean=MK[c].mean(), indie_mean=MI[c].mean(),
                                  d_kpop_minus_indie=(MK[c].mean() - MI[c].mean()) / np.sqrt((MK[c].var() + MI[c].var()) / 2),
                                  p_mannwhitney=mannwhitneyu(MK[c].dropna(), MI[c].dropna())[1]) for c in MIR_COLS])
        CMPM = pd.concat([CMPM, pd.DataFrame([dict(feature='song-to-song audio similarity', construct='standardization (between songs)',
                                                   kpop_mean=np.median(kk_), indie_mean=coh_(Zi),
                                                   d_kpop_minus_indie=np.nan, p_mannwhitney=np.mean(kk_ <= coh_(Zi)))])], ignore_index=True)
        # last row: p = share of random K-pop subsets (same size as indie) that are NOT more coherent than indie

with pd.ExcelWriter(OUT / 'tables' / 'mir_and_typicality.xlsx', engine='xlsxwriter') as xw:
    MSUM.to_excel(xw, sheet_name='mir_summary', index=False); MDESC.to_excel(xw, sheet_name='mir_constructs', index=False)
    TM.to_excel(xw, sheet_name='mir_per_song', index=False); TYP.to_excel(xw, sheet_name='typicality', index=False)
    LOAD.round(3).to_excel(xw, sheet_name='dimensions_loadings'); DIMR.to_excel(xw, sheet_name='dimensions_vs_index', index=False)
    if CMPM is not None: CMPM.to_excel(xw, sheet_name='kpop_vs_indie_MIR', index=False)
display(MSUM); display(MDESC.round(3)); display(TYP.round(3)); display(DIMR.round(3))
if CMPM is not None: display(CMPM.round(3))
print('saved: tables/mir_and_typicality.xlsx', '' if CMPM is not None else '(indie part missing: run cell 10 first)')

In [ ]:
# 12) Reviewer checks (Oct 2026) -> OUT/tables/reviewer_checks.xlsx. Run after cells 9, 10 and 11.
#   (a) gender separability with leave-one-ACT-out: all videos of the held-out act are removed from both class means
#   (b) K-pop vs indie visual coherence with K-pop restricted to 2021+ (the indie corpus covers 2021-2026)
#   (c) tempo distribution and octave errors (half/double tempo folded into 80-160 BPM)
def sep_acc_group(X, lab, grp):
    G = X @ X.T; np.fill_diagonal(G, np.nan); hit = np.zeros(len(X), bool); cls = np.unique(lab)
    for j in range(len(X)):
        sc = []
        for c in cls:
            m = (lab == c) & (grp != grp[j])
            W_ = G[np.ix_(m, m)]; mu, sd = np.nanmean(W_), np.nanstd(W_) + 1e-9
            sc.append((np.nanmean(G[j, m]) - mu) / sd)
        hit[j] = cls[int(np.argmax(sc))] == lab[j]
    return np.array([hit[lab == c].mean() for c in cls])
acts_ = T['artist'].to_numpy()
acc_g = sep_acc_group(V, fem.astype(int), acts_); bal_gg = acc_g.mean()
bal_g = sep_acc(V, fem.astype(int)).mean()   # leave-one-video-out, as in chapter 4 (cell 13 may not have run yet)
act_gender = T.groupby('artist')['gender'].first(); r_ = np.random.default_rng(21); null_gg = []
for _ in range(200):                                   # labels permuted between ACTS, not videos
    perm = pd.Series(r_.permutation(act_gender.to_numpy()), index=act_gender.index)
    null_gg.append(sep_acc_group(V, (T['artist'].map(perm).str.lower() == 'female').to_numpy().astype(int), acts_).mean())
null_gg = np.array(null_gg)
CHK = [('gender separability, leave-one-video-out (as in chapter 4)', f'balanced {bal_g:.3f}'),
       ('gender separability, leave-one-ACT-out', f'female {acc_g[1]:.3f}, male {acc_g[0]:.3f}; balanced {bal_gg:.3f} '
        f'vs act-level permutation null {null_gg.mean():.3f}, p={np.mean(null_gg >= bal_gg):.3f}')]

recent = pd.to_numeric(T['release_year'], errors='coerce').to_numpy() >= 2021
by_act_r = T[recent].groupby('artist')['file_name'].apply(list); k_r = min(len(by_iart), len(by_act_r))
kk_r = np.array([coh(one_per([by_act_r[a_] for a_ in rng.choice(by_act_r.index, k_r, replace=False)], K)) for _ in range(2000)])
ii_r = np.array([coh(one_per([by_iart[a_] for a_ in rng.choice(by_iart.index, k_r, replace=False)], I)) for _ in range(2000)])
CHK += [('K-pop 2021+ only: videos / acts', f'{recent.sum()} / {len(by_act_r)}'),
        (f'K-pop 2021+ coherence, {k_r} videos one per act (median)', f'{np.median(kk_r):.4f}'),
        (f'indie coherence, {k_r} videos one per artist (median)', f'{np.median(ii_r):.4f}'),
        ('P(K-pop 2021+ subset more coherent than indie subset)', f'{np.mean(kk_r[:, None] > ii_r[None, :1000]):.3f}')]

def fold(t):
    t = np.asarray(t, float).copy()
    for _ in range(4):
        t[t < 80] *= 2; t[t >= 160] /= 2
    return t
from scipy.stats import mannwhitneyu
TEMPO = {'K-pop': MK['tempo_bpm'].dropna().to_numpy()}
if 'MI' in globals():
    TEMPO['indie'] = MI['tempo_bpm'].dropna().to_numpy()
for name, t in TEMPO.items():
    f = fold(t)
    CHK.append((f'tempo {name}: mean, SD, median (IQR)', f'{t.mean():.1f}, {t.std(ddof=1):.1f}, {np.median(t):.1f} ({np.percentile(t, 25):.1f} to {np.percentile(t, 75):.1f})'))
    CHK.append((f'tempo {name} folded into 80-160 BPM: mean, SD; share changed by folding', f'{f.mean():.1f}, {f.std(ddof=1):.1f}; {np.mean(f != t):.2f}'))
if 'indie' in TEMPO:
    CHK.append(('tempo K-pop vs indie: Mann-Whitney p, raw / folded',
                f'{mannwhitneyu(TEMPO["K-pop"], TEMPO["indie"]).pvalue:.3f} / {mannwhitneyu(fold(TEMPO["K-pop"]), fold(TEMPO["indie"])).pvalue:.3f}'))
# (d) reproducibility record: exact model revision and library versions (for the appendix)
try:
    from huggingface_hub import HfApi; CHK.append(('model checkpoint (Hugging Face) and commit', f'{MODEL_ID} @ {HfApi().model_info(MODEL_ID).sha}'))
except Exception as e:
    CHK.append(('model checkpoint (Hugging Face) and commit', f'{MODEL_ID} (commit not available offline: {e})'))
import importlib.metadata as im
def ver_(m_):
    try: return im.version(m_)
    except Exception: return 'n/a'
CHK.append(('library versions', ', '.join(f'{m_} {ver_(m_)}' for m_ in ['torch', 'transformers', 'decord', 'librosa', 'scenedetect', 'numpy', 'scipy', 'pandas'])))
CHK = pd.DataFrame(CHK, columns=['check', 'result'])
pd.set_option('display.max_colwidth', None); display(CHK)
CHK.to_excel(OUT / 'tables' / 'reviewer_checks.xlsx', index=False)
print('saved: tables/reviewer_checks.xlsx  -> send this table')

In [ ]:
# 13) FINAL SYNTHESIS of the product side -> OUT/tables/FINAL_product_results.xlsx
#     Run after cells 9, 10 and 11. Collects every result, applies the rules fixed in chapter 3, writes one verdict table.
def rho_p(a, b):
    r, p = spearmanr(a, b, nan_policy='omit'); return r, p
def fmt(r, p): return f'rho={r:+.3f}, p={p:.3f}'
V_ = []
def verdict(q, stat, rule, text): V_.append(dict(question=q, statistic=stat, rule=rule, verdict=text))

r, p = rho_p(T['visual_repetition'], T['index_balanced_z'])
verdict('Index not driven by within-video repetition (averaging check)', fmt(r, p), '|rho| < .30',
        'PASS' if abs(r) < .30 else ('FAIL: report window-level index too' if r <= -.30 else 'CHECK'))
r2, _ = rho_p(T['n_windows'], T['index_balanced_z'])
verdict('Index not driven by video length', f'rho={r2:+.3f}', '|rho| < .30', 'PASS' if abs(r2) < .30 else 'CHECK')
sens = {c: rho_p(T[c], T['index_balanced_z'])[0] for c in ['index_raw_z', 'index_within_gender_z', 'index_knn10_z', 'index_windowlevel_z']}
verdict('Robustness across index versions', ', '.join(f'{k_}={v_:+.2f}' for k_, v_ in sens.items()), 'all rho >= .60',
        'ROBUST' if min(sens.values()) >= .60 else 'PARTLY ROBUST (see lowest version)')
Sf, Sm2, Sx = np.nanmean(S[np.ix_(fem, fem)]), np.nanmean(S[np.ix_(~fem, ~fem)]), np.nanmean(S[np.ix_(fem, ~fem)])
verdict('Two gender types of visual form', f'within F={Sf:.3f}, within M={Sm2:.3f}, between={Sx:.3f}', 'both within > between',
        'TWO TYPES' if min(Sf, Sm2) > Sx else 'NO CLEAR TYPES')
d_raw = (T.loc[fem, 'index_raw'].mean() - T.loc[~fem, 'index_raw'].mean()) / T['index_raw'].std()
d_bal = (T.loc[fem, 'index_balanced'].mean() - T.loc[~fem, 'index_balanced'].mean()) / T['index_balanced'].std()
verdict('Raw gender gap is a composition artifact', f'd raw={d_raw:.2f}, d balanced={d_bal:.2f}', 'balanced |d| clearly smaller',
        'YES' if abs(d_bal) < abs(d_raw) / 2 else 'NO')
try:
    pt = float(TREND.set_index('term').loc['year_frac', 'p']); ct = float(TREND.set_index('term').loc['year_frac', 'coef'])
    verdict('Standardization increases over time (2018-2026)', f'coef/year={ct:+.3f}, p={pt:.3f}', 'coef > 0 and p < .05',
            'SUPPORTED' if (ct > 0 and pt < .05) else ('OPPOSITE DIRECTION' if (ct < 0 and pt < .05) else 'NOT SUPPORTED (no reliable trend)'))
except Exception as e:
    verdict('Standardization increases over time', 'n/a', '', f'not available: {e}')
r, p = rho_p(-maha, T['index_balanced_z'])
verdict('Index tracks typicality of simple features (closeness to corpus centre)', fmt(r, p), 'rho > 0, p < .05',
        'YES' if (r > 0 and p < .05) else 'NO: the index captures something not reducible to these features')
r, p = rho_p(TM['audio_index_balanced_z'], TM['index_balanced_z'])
verdict('Audio and visual standardization go together', fmt(r, p), 'rho > 0, p < .05', 'YES' if (r > 0 and p < .05) else 'NO / INDEPENDENT')
r, p = rho_p(TM['audio_repetition'], TM['audio_index_balanced_z'])
verdict('Within-song repetition goes with between-song standardization', fmt(r, p), 'rho > 0, p < .05', 'YES' if (r > 0 and p < .05) else 'NO: repetition and standardization are distinct')
pk = float(np.mean(kk[:, None] > ii[None, :1000]))
verdict('K-pop visually more homogeneous than Korean indie', f'P(K-pop subset more coherent)={pk:.3f}', '> .95 yes; < .05 opposite',
        'YES' if pk > .95 else ('OPPOSITE' if pk < .05 else 'INDETERMINATE'))
verdict('K-pop and indie visually separable (distinctive convention)', f'LOO accuracy K-pop={acc_k:.2f}, indie={acc_i:.2f}; balanced={bal_ki:.2f} (null {null_ki:.2f}), p={p_ki:.3f}',
        'p < .05 and balanced > .75', 'SEPARABLE' if (p_ki < .05 and bal_ki > .75) else ('PARTLY SEPARABLE' if p_ki < .05 else 'NOT SEPARABLE'))
for c in ['visual_repetition', 'audio_repetition', 'cuts_per_min', 'camera_movement']:
    row = CMP.set_index('feature').loc[c]
    verdict(f'K-pop vs indie: {c}', f'd={row.d_kpop_minus_indie:+.2f}, p_holm={row.p_holm:.3f}', 'p_holm < .05',
            ('K-POP HIGHER' if row.d_kpop_minus_indie > 0 else 'INDIE HIGHER') if row.p_holm < .05 else 'NO RELIABLE DIFFERENCE')
if CMPM is not None:
    cm = CMPM.set_index('feature')
    last = cm.loc['song-to-song audio similarity']
    verdict('K-pop sonically more homogeneous than indie', f'K-pop median subset={last.kpop_mean:.3f}, indie={last.indie_mean:.3f}, share K-pop not more coherent={last.p_mannwhitney:.3f}',
            'share < .05', 'YES' if last.p_mannwhitney < .05 else ('OPPOSITE' if last.p_mannwhitney > .95 else 'INDETERMINATE'))
    mm = cm.drop(index='song-to-song audio similarity').dropna(subset=['p_mannwhitney'])
    o = np.argsort(mm['p_mannwhitney'].to_numpy()); ph = np.empty(len(o)); run = 0
    for i_, j_ in enumerate(o): run = max(run, (len(o) - i_) * mm['p_mannwhitney'].iloc[j_]); ph[j_] = min(1, run)
    mm = mm.assign(p_holm=ph)
    for f_, row in mm.iterrows():
        verdict(f'MIR K-pop vs indie: {f_} [{row.construct}]', f'd={row.d_kpop_minus_indie:+.2f}, p_holm={row.p_holm:.3f}', 'p_holm < .05',
                ('K-POP HIGHER' if row.d_kpop_minus_indie > 0 else 'INDIE HIGHER') if row.p_holm < .05 else 'no reliable difference')
else:
    verdict('MIR K-pop vs indie', 'n/a', '', 'run cell 11 after cell 10, then this cell again')
verdict('V1: index agrees with human template-conformity coding', 'pending', 'rho >= .30', 'PENDING (cell 18 after human coding)')
# ---- additional chapter-3 requirements (3-3-2, 3-3-3, 3-3-4, 3-4-4, 3-4-5, 3-8) ----
q = T['index_balanced'].quantile([.25, .5, .75]); iqr_z = T['index_balanced_z'].quantile(.75) - T['index_balanced_z'].quantile(.25)
verdict('Distribution of the index (RQ2)', f'Q1={q[.25]:.4f}, median={q[.5]:.4f}, Q3={q[.75]:.4f}, IQR(z)={iqr_z:.2f}', 'range not compressed',
        'OK' if iqr_z > .8 else 'COMPRESSED: use tertiles for excerpt selection')
r, p = rho_p(T['resultant_R'], T['index_balanced_z'])
verdict('Index not driven by window diversity (resultant length)', fmt(r, p), '|rho| < .30', 'PASS' if abs(r) < .30 else 'CHECK')
# gender: within-group coherence F vs M (permutation) and separability of the two types
cohF, cohM = np.nanmean(S[np.ix_(fem, fem)]), np.nanmean(S[np.ix_(~fem, ~fem)]); diffs = []
prng = np.random.default_rng(5)
for _ in range(2000):
    f_ = prng.permutation(fem); diffs.append(np.nanmean(S[np.ix_(f_, f_)]) - np.nanmean(S[np.ix_(~f_, ~f_)]))
p_perm = np.mean(np.abs(diffs) >= abs(cohF - cohM))
verdict('Female and male videos equally coherent inside their type', f'F={cohF:.3f}, M={cohM:.3f}, permutation p={p_perm:.3f}', 'p >= .05 equal',
        'EQUAL' if p_perm >= .05 else ('FEMALE MORE COHERENT' if cohF > cohM else 'MALE MORE COHERENT'))
(acc_m, acc_f), bal_g, p_g, null_g = sep_test(V, fem.astype(int))
verdict('Gender types separable in the model space', f'LOO accuracy female={acc_f:.2f}, male={acc_m:.2f}; balanced={bal_g:.2f} (null {null_g:.2f}), p={p_g:.3f}',
        'p < .05 and balanced > .75', 'SEPARABLE' if (p_g < .05 and bal_g > .75) else ('PARTLY SEPARABLE' if p_g < .05 else 'NOT SEPARABLE'))
# time: cluster bootstrap of the slope (resampling acts) + generations
TT2 = T.copy(); TT2['yr'] = pd.to_datetime(TT2['mv_release_date']).dt.year + pd.to_datetime(TT2['mv_release_date']).dt.dayofyear / 365.25
acts2 = TT2['artist'].unique(); brng = np.random.default_rng(9); slopes = []
for _ in range(2000):
    smp = pd.concat([TT2[TT2.artist == a_] for a_ in brng.choice(acts2, len(acts2))])
    X_ = np.c_[np.ones(len(smp)), smp['yr'], (smp['gender'].str.lower() == 'female')]
    slopes.append(np.linalg.lstsq(X_, smp['index_balanced_z'].to_numpy(), rcond=None)[0][1])
lo, hi = np.percentile(slopes, [2.5, 97.5])
verdict('Time trend, act-level cluster bootstrap (per year, z units)', f'95% CI {lo:+.3f} to {hi:+.3f}', 'CI above 0', 'INCREASING' if lo > 0 else ('DECREASING' if hi < 0 else 'NO RELIABLE TREND'))
from scipy.stats import kruskal
gens = [g_['index_balanced_z'].to_numpy() for _, g_ in T.groupby('generation_by_debut_rule') if len(g_) >= 3]
GEN = T.groupby('generation_by_debut_rule')['index_balanced_z'].agg(['count', 'mean', 'median']).reset_index()
if len(gens) >= 2:
    hk = kruskal(*gens); verdict('Index differs between idol generations', ', '.join(f'{r_.generation_by_debut_rule}: {r_["mean"]:+.2f} (n={r_["count"]})' for _, r_ in GEN.iterrows()) + f'; Kruskal p={hk.pvalue:.3f}',
                                'p < .05', 'DIFFERS' if hk.pvalue < .05 else 'NO RELIABLE DIFFERENCE')
# audio side: gender split of sound, raw-reference sensitivity, chorus candidate agreement
SAf, SAm, SAx = np.nanmean(SA[np.ix_(fem, fem)]), np.nanmean(SA[np.ix_(~fem, ~fem)]), np.nanmean(SA[np.ix_(fem, ~fem)])
verdict('Does sound also split by group gender? (3-4-5)', f'within F={SAf:.3f}, within M={SAm:.3f}, between={SAx:.3f}', 'both within > between',
        'YES: sound also splits' if min(SAf, SAm) > SAx else 'NO: the gender axis of the visual index is about bodies/performers, not form of sound')
r, p = rho_p(zs(np.nanmean(SA, 1)), TM['audio_index_balanced_z'])
verdict('Audio index robust to reference choice (raw vs balanced)', fmt(r, p), 'rho >= .60', 'ROBUST' if r >= .60 else 'CHECK')
r, p = rho_p(TM['pseudo_individualization_z'], TM['index_balanced_z'])
verdict('Pseudo-individualization (sound) vs visual standardization (exploratory)', fmt(r, p), 'descriptive', 'see chapter 4 discussion')
try:
    chk2 = pd.read_excel(OUT / 'tables' / 'chorus_check.xlsx'); chk2 = chk2[chk2['chorus_start_verified'].notna()]
    if len(chk2):
        best = chk2[['chorus_cand_1', 'chorus_cand_2', 'chorus_cand_3']].sub(chk2['chorus_start_verified'], axis=0).abs().min(1)
        verdict('Chorus candidates vs verified chorus start (validity check)', f'n={len(chk2)}, within 5 s: {np.mean(best <= 5):.2f}, median error {best.median():.1f} s',
                'descriptive', 'reported')
    else:
        verdict('Chorus candidates vs verified chorus start', 'pending', '', 'fill chorus_check.xlsx, then re-run')
except Exception as e:
    verdict('Chorus candidates vs verified chorus start', 'pending', '', f'not available: {e}')
# ranking and 2-D map (3-8)
RANK = T[['file_name', 'artist', 'gender', 'release_year', 'index_balanced_z', 'index_raw_z', 'index_within_gender_z']].sort_values('index_balanced_z', ascending=False)
Ue, se, _ = np.linalg.svd(V - V.mean(0), full_matrices=False); P2 = Ue[:, :2] * se[:2]
PI2 = (VI - V.mean(0)) @ _[:2].T
plt.figure(figsize=(7, 6))
plt.scatter(P2[fem, 0], P2[fem, 1], s=14, c='tab:red', label='K-pop female'); plt.scatter(P2[~fem, 0], P2[~fem, 1], s=14, c='tab:blue', label='K-pop male')
plt.scatter(PI2[:, 0], PI2[:, 1], s=14, c='tab:green', marker='^', label='Korean indie'); plt.legend(); plt.title('2-D map of the model space (PCA of video vectors)')
plt.tight_layout(); plt.savefig(OUT / 'figures' / 'map_kpop_indie.png', dpi=150); plt.show()
VERD = pd.DataFrame(V_)
with pd.ExcelWriter(OUT / 'tables' / 'FINAL_product_results.xlsx', engine='xlsxwriter') as xw:
    VERD.to_excel(xw, sheet_name='verdicts', index=False); RANK.to_excel(xw, sheet_name='ranking', index=False)
    SUM.to_excel(xw, sheet_name='kpop_summary', index=False); GT.to_excel(xw, sheet_name='gender_types', index=False)
    TREND.to_excel(xw, sheet_name='time_trend', index=False); GEN.to_excel(xw, sheet_name='generations', index=False)
    REF.to_excel(xw, sheet_name='indie_visual', index=False); CMP.to_excel(xw, sheet_name='indie_features', index=False)
    MSUM.to_excel(xw, sheet_name='mir_summary', index=False); MDESC.to_excel(xw, sheet_name='mir_constructs', index=False)
    TM.to_excel(xw, sheet_name='mir_per_song', index=False); TYP.to_excel(xw, sheet_name='typicality', index=False)
    DIMR.to_excel(xw, sheet_name='dimensions', index=False); LOAD.round(3).to_excel(xw, sheet_name='dimension_loadings')
    if CMPM is not None: CMPM.to_excel(xw, sheet_name='indie_mir', index=False)
display(VERD)
print('saved: tables/FINAL_product_results.xlsx, figures/map_kpop_indie.png  -> send the Excel file')

### محرک‌های مطالعه‌ی ردیابی چشم (سلول‌های ۱۴ تا ۱۷)
قاعده‌ی انتخاب فقط شاخص است و هیچ ویژگی ادراکی هم‌تراز نمی‌شود (فصل ۳، ۳-۹-۲). کی‌پاپ: درون هر جنسیت دو قطعه با بالاترین و دو با پایین‌ترین شاخص قطعه، از هر گروه یکی. ایندی: دو ویدئو با بیشترین و دو با کمترین شباهت به کی‌پاپ، در هر سر یکی با خواننده‌ی زن و یکی با خواننده‌ی مرد. کلیپ‌ها ۲۰ ثانیه، ۱۰۸۰، H.264، بلندی ‎−۱۶ LUFS‎ و با نام بی‌نشان‌اند.


In [ ]:
# 14) Step 4 prep: chorus-excerpt index + selection of 8 excerpts for the eye-tracking study
EXC = 20.0
_src = OUT / 'tables' / ('chorus_shortlist.xlsx' if (OUT / 'tables' / 'chorus_shortlist.xlsx').exists() else 'chorus_check.xlsx')
chk = pd.read_excel(_src); chk = chk[chk['chorus_start_verified'].notna()]; print('verified choruses from', _src.name, ':', len(chk))
if 'exclude_for_eyetracking' in chk: chk = chk[chk['exclude_for_eyetracking'].isna() | (chk['exclude_for_eyetracking'].astype(str).str.strip() == '')]
exc_v, exc_feat = {}, {}
for _, r in chk.iterrows():
    x, c = r['file_name'], float(r['chorus_start_verified'])
    if x not in K: continue
    st, en, W = K[x]['starts'], K[x]['ends'], K[x]['W']
    sel = (np.clip(np.minimum(en, c + EXC) - np.maximum(st, c), 0, None) / (en - st)) >= 0.5
    if not sel.any(): continue
    exc_v[x] = l2n(W[sel].mean(0))
    v = np.load(K[x]['dir'] / 'visual.npz'); sh = json.loads((K[x]['dir'] / 'shots.json').read_text(encoding='utf-8'))['shots']
    inf = (v['t_frame'] >= c) & (v['t_frame'] < c + EXC); inm = (v['t_motion'] >= c) & (v['t_motion'] < c + EXC)
    exc_feat[x] = dict(chorus_start=c, ex_luminance=v['luminance'][inf].mean(), ex_contrast=v['contrast'][inf].mean(),
                       ex_motion=v['motion'][inm].mean(), ex_cuts_per_min=60 * sum(1 for s, e in sh if c < s < c + EXC) / EXC,
                       ex_closeup_share=float((v['max_face_area'][inf] >= 0.05).mean()))
EN = [x for x in names if x in exc_v]; VE = np.stack([exc_v[x] for x in EN])
# excerpt index = balanced similarity of the chorus excerpt to all 105 full videos (own video excluded),
# expressed in the z units of the video-level index so the two are directly comparable
SE = VE @ V.T
for i, x in enumerate(EN): SE[i, names.index(x)] = np.nan
_mu, _sd = T['index_balanced'].mean(), T['index_balanced'].std()
EX = pd.DataFrame({'file_name': EN, 'excerpt_index_z': (balanced(SE, fem) - _mu) / _sd})
EX = EX.merge(pd.DataFrame([dict(file_name=x, **exc_feat[x]) for x in EN]), on='file_name').merge(T[['file_name', 'artist', 'gender', 'index_balanced_z']], on='file_name')
EX = EX.merge(chk[['file_name', 'exclude_for_eyetracking']], on='file_name')
from scipy.stats import spearmanr
_r = spearmanr(EX['excerpt_index_z'], EX['index_balanced_z'])
print(f'chorus representativeness: rho(excerpt index, whole-video index) = {_r[0]:+.2f} (p={_r[1]:.3f}, n={len(EX)})')
# excerpt_index_z is on the whole-video scale: a 20-s excerpt averages ~4 windows, so it is always less similar to the
# corpus than a whole video (~50 windows) and every value comes out negative. excerpt_z_within re-standardises among excerpts.
EX['excerpt_z_within'] = zs(EX['excerpt_index_z'])
display(EX[['file_name', 'gender', 'index_balanced_z', 'excerpt_index_z', 'excerpt_z_within']].sort_values('excerpt_z_within', ascending=False).round(2))
if 'level' in chk: EX = EX.merge(chk[['file_name', 'level']], on='file_name')
EX = EX[EX['exclude_for_eyetracking'].fillna('').astype(str).str.strip() == ''].reset_index(drop=True)
from itertools import combinations
MATCH = ['ex_luminance', 'ex_contrast', 'ex_motion', 'ex_cuts_per_min']; sdv = EX[MATCH].std()
def pools(q):
    out = {}
    for g in ['female', 'male']:
        s_ = EX[EX.gender.str.lower() == g]
        if 'level' in EX:   # shortlist: high/low cells were fixed when the shortlist was built
            out[g] = (s_[s_['level'] == 'high'], s_[s_['level'] == 'low']); continue
        lo, hi = s_['excerpt_index_z'].quantile([q, 1 - q])
        out[g] = (s_[s_['excerpt_index_z'] >= hi], s_[s_['excerpt_index_z'] <= lo])
    return out
def pairs(pool): return [c for c in combinations(pool.index, 2) if pool.loc[list(c), 'artist'].nunique() == 2]
best = None
for q, label in [(0.25, 'quartiles'), (1 / 3, 'tertiles')]:
    P_ = pools(q)
    for fh in pairs(P_['female'][0]):
        for fl in pairs(P_['female'][1]):
            for mh in pairs(P_['male'][0]):
                for ml in pairs(P_['male'][1]):
                    ch = list(fh) + list(fl) + list(mh) + list(ml)
                    if EX.loc[ch, 'artist'].nunique() < 8: continue
                    hi_, lo_ = EX.loc[list(fh) + list(mh)], EX.loc[list(fl) + list(ml)]
                    cost = float((np.abs(hi_[MATCH].mean() - lo_[MATCH].mean()) / sdv).sum())
                    if best is None or cost < best[0]: best = (cost, ch, label)
    if best: break
if best:
    SEL = EX.loc[best[1]].copy(); SEL['level'] = ['high'] * 2 + ['low'] * 2 + ['high'] * 2 + ['low'] * 2
    SEL['excerpt_end'] = SEL['chorus_start'] + EXC
    gap = (SEL[SEL.level == 'high']['excerpt_index_z'].mean() - SEL[SEL.level == 'low']['excerpt_index_z'].mean()) / EX['excerpt_index_z'].std()
    print(f'selection rule: {best[2]} | matching cost {best[0]:.3f} | index gap high-low = {gap:.2f} SD (needs >= 1)')
    with pd.ExcelWriter(OUT / 'tables' / 'eyetracking_selection.xlsx', engine='xlsxwriter') as xw:
        SEL.to_excel(xw, sheet_name='selected_8', index=False); EX.to_excel(xw, sheet_name='all_excerpts', index=False)
    display(SEL[['file_name', 'gender', 'level', 'excerpt_index_z', 'chorus_start'] + MATCH].round(3))
    # cut the 8 clips (original quality) for the eye-tracker
    clip_dir = OUT / 'eyetracking_clips'; clip_dir.mkdir(exist_ok=True)
    src = {p.name: p for p in list_videos(KPOP_DIR)}
    for i, r in SEL.reset_index(drop=True).iterrows():
        outc = clip_dir / f"ET{i+1}_{r['level']}_{slug(r['file_name'])}.mp4"
        subprocess.run([FFMPEG, '-loglevel', 'error', '-y', '-ss', str(r['chorus_start']), '-i', str(src[r['file_name']]), '-t', str(EXC),
                        '-c:v', 'libx264', '-crf', '16', '-c:a', 'aac', '-b:a', '192k', str(outc)], check=True)
    print('8 clips saved in', clip_dir)
else:
    print('No admissible set even with tertiles. Send me this message.')

In [ ]:
# 15) blind-named excerpts of every verified shortlist chorus, for the human rating page
import random
_sl = pd.read_excel(OUT / 'tables' / 'chorus_shortlist.xlsx'); _sl = _sl[_sl['chorus_start_verified'].notna()]
_sl = _sl[_sl['exclude_for_eyetracking'].isna() | (_sl['exclude_for_eyetracking'].astype(str).str.strip() == '')].reset_index(drop=True)
rdir = OUT / 'web_clips' / 'rate'; rdir.mkdir(parents=True, exist_ok=True)
src = {p.name: p for p in list_videos(KPOP_DIR)}
HI = KPOP_DIR / '1080'
def _k(name): return re.sub(r'[^0-9a-z]', '', unicodedata.normalize('NFC', name).lower().rsplit('_', 1)[0])
hi = {_k(p.stem): p for p in sorted(HI.glob('*')) if p.suffix.lower() in VIDEO_EXT} if HI.exists() else {}
rids = [f'R{i:02d}' for i in range(1, len(_sl) + 1)]; random.Random(7).shuffle(rids)
rk = []
for rid, (_, r) in zip(rids, _sl.iterrows()):
    use = hi.get(_k(Path(r['file_name']).stem), src[r['file_name']])
    subprocess.run([FFMPEG, '-loglevel', 'error', '-y', '-ss', str(r['chorus_start_verified']), '-i', str(use), '-t', '20',
                    '-vf', r'scale=-2:min(ih\,720)', '-c:v', 'libx264', '-preset', 'medium', '-crf', '22', '-pix_fmt', 'yuv420p',
                    '-af', 'loudnorm=I=-16:TP=-1.5:LRA=11', '-c:a', 'aac', '-b:a', '128k', '-movflags', '+faststart', str(rdir / f'{rid}.mp4')], check=True)
    rk.append(dict(rate_id=rid, file_name=r['file_name'], gender=r['gender'], level=r['level'], chorus_start=r['chorus_start_verified']))
RK = pd.DataFrame(rk).sort_values('rate_id'); RK.to_excel(OUT / 'web_clips' / 'rate_key.xlsx', index=False)
print(len(RK), 'rating clips in', rdir, '| ids:', ', '.join(RK.rate_id))

In [ ]:
# 16) web-ready, blind-named clips for the online eye-tracker
import random
SELX = pd.read_excel(OUT / 'tables' / 'eyetracking_selection.xlsx', sheet_name='selected_8')
web = OUT / 'web_clips'; (web / 'clips').mkdir(parents=True, exist_ok=True)
src = {p.name: p for p in list_videos(KPOP_DIR)}
# prefer the 1080p copies in 'kpop Corpus/1080' (same YouTube video, higher resolution), matched by title
HI = KPOP_DIR / '1080'
def _k(name): return re.sub(r'[^0-9a-z]', '', unicodedata.normalize('NFC', name).lower().rsplit('_', 1)[0])
hi = {_k(p.stem): p for p in sorted(HI.glob('*')) if p.suffix.lower() in VIDEO_EXT} if HI.exists() else {}
def dur(p):
    c = cv2.VideoCapture(str(p)); n, f = c.get(cv2.CAP_PROP_FRAME_COUNT), c.get(cv2.CAP_PROP_FPS); c.release(); return n / f if f else float('nan')
ids = [f'C{i}' for i in range(1, len(SELX) + 1)]; random.Random(2026).shuffle(ids)
key = []
for cid, (_, r) in zip(ids, SELX.iterrows()):
    out = web / 'clips' / f'{cid}.mp4'
    lo = src[r['file_name']]; hp = hi.get(_k(Path(r['file_name']).stem))
    # use the 1080p copy only if it is the same video (durations within 1 s), otherwise the corpus file
    use = hp if hp is not None and abs(dur(hp) - dur(lo)) < 1.0 else lo
    if hp is None: print('no 1080p copy found for', r['file_name'])
    elif use is lo: print('1080p copy has a different length, using 720p for', r['file_name'])
    # H.264 (plays in every browser); CRF 18 = visually near-lossless; cap keeps each clip under Cloudflare's 25 MB file limit
    subprocess.run([FFMPEG, '-loglevel', 'error', '-y', '-ss', str(r['chorus_start']), '-i', str(use), '-t', '20',
                    '-vf', r'scale=-2:min(ih\,1080)', '-c:v', 'libx264', '-preset', 'slow', '-crf', '18', '-profile:v', 'high', '-pix_fmt', 'yuv420p',
                    '-maxrate', '8000k', '-bufsize', '16000k',
                    '-af', 'loudnorm=I=-16:TP=-1.5:LRA=11', '-c:a', 'aac', '-b:a', '160k', '-movflags', '+faststart', str(out)], check=True)
    key.append(dict(clip_id=cid, file_name=r['file_name'], source=use.name, gender=r['gender'], level=r['level'],
                    excerpt_index_z=r['excerpt_index_z'], chorus_start=r['chorus_start'], size_mb=round(out.stat().st_size / 1e6, 1)))
KEY = pd.DataFrame(key).sort_values('clip_id'); KEY.to_excel(web / 'clip_key.xlsx', index=False)
display(KEY[['clip_id', 'source', 'gender', 'level', 'size_mb']])
print('total MB:', KEY.size_mb.sum(), '| upload folder:', web / 'clips', '| keep private:', web / 'clip_key.xlsx')

In [ ]:
# 17) indie excerpts C9-C12 (cut from the 1080p copies) + excerpt index of all 12 eye-tracking clips on one scale
I = load_corpus('kindie')
import random
EXC = 20.0
def _k(name): return re.sub(r'[^0-9a-z]', '', unicodedata.normalize('NFC', name).lower().rsplit('_', 1)[0])
def dur(p):
    c = cv2.VideoCapture(str(p)); n, f = c.get(cv2.CAP_PROP_FRAME_COUNT), c.get(cv2.CAP_PROP_FPS); c.release(); return n / f if f else float('nan')
def ex_vec(R, x, c):
    st, en, W = R[x]['starts'], R[x]['ends'], R[x]['W']
    sel = (np.clip(np.minimum(en, c + EXC) - np.maximum(st, c), 0, None) / (en - st)) >= 0.5
    return l2n(W[sel].mean(0))
def ex_feat(R, x, c):
    v = np.load(R[x]['dir'] / 'visual.npz'); sh = json.loads((R[x]['dir'] / 'shots.json').read_text(encoding='utf-8'))['shots']
    inf = (v['t_frame'] >= c) & (v['t_frame'] < c + EXC); inm = (v['t_motion'] >= c) & (v['t_motion'] < c + EXC)
    return dict(ex_luminance=v['luminance'][inf].mean(), ex_contrast=v['contrast'][inf].mean(), ex_motion=v['motion'][inm].mean(),
                ex_cuts_per_min=60 * sum(1 for s, e in sh if c < s < c + EXC) / EXC, ex_closeup_share=float((v['max_face_area'][inf] >= 0.05).mean()))
_mu, _sd = T['index_balanced'].mean(), T['index_balanced'].std()
def ex_index(vec, own=None):   # balanced similarity of an excerpt to the 105 K-pop videos (own video excluded), in whole-video z units
    s = vec @ V.T
    if own is not None: s[names.index(own)] = np.nan
    return float((balanced(s[None, :], fem)[0] - _mu) / _sd)
# the 8 K-pop clips already chosen
S8 = pd.read_excel(OUT / 'tables' / 'eyetracking_selection.xlsx', sheet_name='selected_8')
rows = [dict(file_name=r['file_name'], corpus='kpop', gender=r['gender'], level=r['level'], chorus_start=float(r['chorus_start']),
             excerpt_index_z=ex_index(ex_vec(K, r['file_name'], float(r['chorus_start'])), r['file_name']),
             **ex_feat(K, r['file_name'], float(r['chorus_start']))) for _, r in S8.iterrows()]
# the 4 indie clips
IC = pd.read_excel(OUT / 'tables' / 'indie_chorus.xlsx'); IC = IC[(IC['use'] == 1) & IC['chorus_start_verified'].notna()]
assert len(IC) == 4, f'indie_chorus.xlsx needs exactly 4 rows with use=1 and a chorus start (found {len(IC)})'
missing = [x for x in IC['file_name'] if x not in I]; assert not missing, f'not in the indie corpus: {missing}'
for _, r in IC.iterrows():
    c = float(r['chorus_start_verified'])
    rows.append(dict(file_name=r['file_name'], corpus='indie', gender=r['singer_gender'], level='indie_' + r['side'], chorus_start=c,
                     excerpt_index_z=ex_index(ex_vec(I, r['file_name'], c)), **ex_feat(I, r['file_name'], c)))
A12 = pd.DataFrame(rows)
display(A12[['corpus', 'level', 'gender', 'excerpt_index_z', 'ex_cuts_per_min', 'ex_luminance']].sort_values('excerpt_index_z', ascending=False).round(2))
# cut C9-C12 (blind names, same encoding as 12b)
web = OUT / 'web_clips'; (web / 'clips').mkdir(parents=True, exist_ok=True)
HI_I = INDIE_DIR / '1080'
hi = {_k(p.stem): p for p in sorted(HI_I.glob('*')) if p.suffix.lower() in VIDEO_EXT} if HI_I.exists() else {}
src = {p.name: p for p in list_videos(INDIE_DIR)}
ids = ['C9', 'C10', 'C11', 'C12']; random.Random(2027).shuffle(ids)
key = []
for cid, (_, r) in zip(ids, A12[A12.corpus == 'indie'].iterrows()):
    out = web / 'clips' / f'{cid}.mp4'
    lo = src[r['file_name']]; hp = hi.get(_k(Path(r['file_name']).stem))
    use = hp if hp is not None and abs(dur(hp) - dur(lo)) < 1.0 else lo
    if hp is None: print('no 1080p copy found for', r['file_name'])
    elif use is lo: print('1080p copy has a different length, using 720p for', r['file_name'])
    subprocess.run([FFMPEG, '-loglevel', 'error', '-y', '-ss', str(r['chorus_start']), '-i', str(use), '-t', '20',
                    '-vf', r'scale=-2:min(ih\,1080)', '-c:v', 'libx264', '-preset', 'slow', '-crf', '18', '-profile:v', 'high', '-pix_fmt', 'yuv420p',
                    '-maxrate', '8000k', '-bufsize', '16000k',
                    '-af', 'loudnorm=I=-16:TP=-1.5:LRA=11', '-c:a', 'aac', '-b:a', '160k', '-movflags', '+faststart', str(out)], check=True)
    key.append(dict(clip_id=cid, file_name=r['file_name'], source=use.name, gender=r['gender'], level=r['level'],
                    excerpt_index_z=r['excerpt_index_z'], chorus_start=r['chorus_start'], size_mb=round(out.stat().st_size / 1e6, 1)))
old = pd.read_excel(web / 'clip_key.xlsx'); old = old[~old['clip_id'].isin(ids)]
KEY = pd.concat([old, pd.DataFrame(key)]).sort_values('clip_id', key=lambda s: s.str[1:].astype(int))
KEY.to_excel(web / 'clip_key.xlsx', index=False)
A12.merge(KEY[['file_name', 'clip_id']], on='file_name', how='left').to_excel(OUT / 'tables' / 'eyetracking_selection_12.xlsx', index=False)
display(pd.DataFrame(key)[['clip_id', 'source', 'gender', 'size_mb']])
print('upload folder:', web / 'clips', '(C1-C12) | keep private:', web / 'clip_key.xlsx')

### پس از کدگذاری انسانی (سلول ۱۸)
پایایی میان کدگذاران (آلفای کریپندورف، حداقل ۰٫۶۷ برای هر مقوله) و روایی شاخص (V1: همبستگی رتبه‌ای شاخص با نیم‌رخ انطباق با قالب، حداقل ۰٫۳۰ با فاصله‌ی اطمینان). نتیجه‌ی V1 تفسیر نتایج نگاه را تعیین می‌کند (OSF، بخش ۵-۸).


In [ ]:
# 18) After human coding: reliability (Krippendorff alpha) + validity of the index (V1)
%pip install -q krippendorff
import krippendorff
CODES = ['shot', 'performance', 'setting', 'n_performers', 'camera_gaze', 'song_section', 'point_move']   # coding manual v2.0
c1 = pd.read_excel(OUT / 'tables' / 'coding_coder1.xlsx', sheet_name='Segment_coding')
c2 = pd.read_excel(OUT / 'tables' / 'coding_coder2.xlsx', sheet_name='Segment_coding')
key = ['code_id', 'segment_no']
both = c1.merge(c2, on=key, suffixes=('_1', '_2'))
both = both[both[[f'{c}_1' for c in CODES]].notna().any(axis=1) & both[[f'{c}_2' for c in CODES]].notna().any(axis=1)]
REL = pd.DataFrame([dict(code=c, n_units=len(both),
                         alpha=krippendorff.alpha(reliability_data=both[[f'{c}_1', f'{c}_2']].T.to_numpy(dtype=float), level_of_measurement='nominal'))
                    for c in CODES])
allc = pd.concat([c1, c2]).dropna(subset=['shot']).drop_duplicates(key)
blind = pd.read_excel(OUT / 'tables' / 'coding_sheet_BLIND.xlsx', sheet_name='videos')
prof = []
for cid, g in allc.groupby('code_id'):
    row = {'code_id': cid}
    for c in ['shot', 'performance', 'setting', 'n_performers', 'camera_gaze']:
        for v_, share in g[c].value_counts(normalize=True).items(): row[f'{c}={int(v_)}'] = share
    prof.append(row)
P = pd.DataFrame(prof).fillna(0).merge(blind, on='code_id').merge(T[['file_name', 'gender', 'index_balanced_z']], on='file_name')
X = l2n(P.filter(like='=').to_numpy(float)); Sp = X @ X.T; np.fill_diagonal(Sp, np.nan)
P['template_conformity_z'] = zs(balanced(Sp, (P['gender'].str.lower() == 'female').to_numpy()))
rho, p = spearmanr(P['template_conformity_z'], P['index_balanced_z'])
print(REL.round(3)); print(f'\nV1 validity: rho(template conformity, balanced index) = {rho:+.2f} (p={p:.3f}); pre-registered: positive and >= .30')
with pd.ExcelWriter(OUT / 'tables' / 'coding_validity.xlsx', engine='xlsxwriter') as xw:
    REL.to_excel(xw, sheet_name='reliability', index=False); P.to_excel(xw, sheet_name='profiles', index=False)
    pd.DataFrame({'rho': [rho], 'p': [p], 'rule': ['V1 supported if rho >= .30']}).to_excel(xw, sheet_name='V1', index=False)
# video level: agreement on the 6 shared videos + template-element count as a second convergent check (exploratory)
v1 = pd.read_excel(OUT / 'tables' / 'coding_coder1.xlsx', sheet_name='Video_level'); v2 = pd.read_excel(OUT / 'tables' / 'coding_coder2.xlsx', sheet_name='Video_level')
EL = [c for c in v1.columns if c.startswith('el_')]; VC = ['outfits', 'sets'] + EL + ['body_display', 'address_mode']
vb = v1.merge(v2, on='code_id', suffixes=('_1', '_2'))
VAGREE = pd.DataFrame([dict(code=c, n_videos=int(vb[[f'{c}_1', f'{c}_2']].notna().all(1).sum()),
                            exact_agreement=float((vb[f'{c}_1'] == vb[f'{c}_2'])[vb[[f'{c}_1', f'{c}_2']].notna().all(1)].mean())) for c in VC])
vall = pd.concat([v1, v2]).drop_duplicates('code_id').merge(blind, on='code_id').merge(T[['file_name', 'gender', 'index_balanced_z']], on='file_name')
vall['template_elements'] = vall[EL].sum(1, min_count=1)
r_el, p_el = spearmanr(vall['template_elements'], vall['index_balanced_z'], nan_policy='omit')
print(VAGREE.round(2)); print(f'template elements vs index (exploratory): rho={r_el:+.2f}, p={p_el:.3f}')
with pd.ExcelWriter(OUT / 'tables' / 'coding_validity_video_level.xlsx', engine='xlsxwriter') as xw:
    VAGREE.to_excel(xw, sheet_name='agreement_6_videos', index=False); vall.to_excel(xw, sheet_name='video_level_all', index=False)
    pd.DataFrame({'rho': [r_el], 'p': [p_el]}).to_excel(xw, sheet_name='template_elements_vs_index', index=False)